In [ ]:
!git clone https://github.com/Simongazze/Tesina_de_grado.git
%cd Tesina_de_grado
!ls

Cloning into 'Tesina_de_grado'...
remote: Enumerating objects: 747, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 747 (delta 74), reused 83 (delta 34), pack-reused 621 (from 1)
Receiving objects: 100% (747/747), 39.32 MiB | 22.76 MiB/s, done.
Resolving deltas: 100% (573/573), done.
/content/Tesina_de_grado
 acciones.xlsx
 Anteproyecto-de-tesina---Simón-Gazze.pdf
'Anteproyecto de tesina - Simón Gazze.Rmd'
 cv_lista.RDS
 data
 datos_01.xlsx
'Desarrollo teórico.Rmd'
 Desarrollo---Tesina-de-grado.html
'Desarrollo - Tesina de grado.R'
'Desarrollo - Tesina de grado.Rmd'
 freq_sim.RDS
 Generación-y-preprocesamiento-de-la-base-de-datos.pdf
'Generación y preprocesamiento de la base de datos.Rmd'
 header.tex
 img
 logo-UNR.jpg
 Logo-unr.png
 match_data.xlsx
 modelo_lasso.RDS
 modelo_multi_enet.RDS
 modelo_multi_lasso.RDS
 modelo_multi_ridge.RDS
 modelo_ridge.RDS
 pbp_crudo.R
 plus_minus_por_equipo_1.RDS
 plus_minus_por_equipo

In [ ]:
# Instalar Google Chrome y ChromeDriver
!apt-get update -y
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get -fy install -y
!pip install selenium webdriver-manager


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,300 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,005 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,607 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,451 kB]
Hit:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy

In [ ]:
import pandas as pd
import re # Carta de trio
import json # Box score
import time # Box score

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

from tqdm import tqdm

pd.set_option('display.max_columns', None)

Link de los partidos

In [ ]:

# Nombre del archivo de texto que contiene el HTML
file_name = "data/elemento_html_partidos_temporada_24_25.txt"

# Diccionario para almacenar los datos de los partidos
match_data = {}
try:
    # 1. Leer el contenido del archivo .txt
    with open(file_name, "r", encoding="utf-8") as f:
        html_content = f.read()

    # 2. Crear un objeto BeautifulSoup para parsear el HTML
    soup = BeautifulSoup(html_content, "html.parser")

    # 3. Buscar todas las filas de la tabla (<tr>)
    filas_partidos = soup.find_all("tr", role="row")

    print(f"Extrayendo datos de {len(filas_partidos)} partidos...")

    for fila in filas_partidos:
        # Extraer los datos de las celdas (<td>) de cada fila
        celdas = fila.find_all("td")

        # Asegurarse de que la fila tiene la estructura esperada
        if len(celdas) > 8:
            # Extraer los datos por su índice de celda
            fecha_hora = celdas[0].get_text(strip=True)[-17:]
            nombre_local = celdas[1].get_text(strip=True)
            puntos_local = celdas[3].get_text(strip=True)
            puntos_visita = celdas[4].get_text(strip=True)
            nombre_visita = celdas[6].get_text(strip=True)

            # El link está dentro de la celda en el índice 8 (anteriormente 9)
            link_tag = celdas[8].find("a", href=True)
            link_estadisticas = link_tag.get('href') if link_tag else None

            # Usar una combinación única como clave del diccionario
            match_key = f"{nombre_local} vs {nombre_visita} ({fecha_hora})"

            # Guardar los datos en el diccionario
            match_data[match_key] = {
                "nombre_local": nombre_local,
                "puntos_local": puntos_local,
                "nombre_visita": nombre_visita,
                "puntos_visita": puntos_visita,
                "link_estadisticas": link_estadisticas
            }

    print("\nDiccionario de datos de partidos creado.")

    # Mostrar el diccionario para su verificación
    display(match_data)

except FileNotFoundError:
    print(f"Error: No se encontró el archivo '{file_name}'. Asegúrate de que el archivo existe en la ruta correcta.")
except Exception as e:
    print(f"Ocurrió un error al procesar el archivo: {e}")

Extrayendo datos de 380 partidos...

Diccionario de datos de partidos creado.


{'ATENAS (C) vs BOCA (007/10/2024 22:10)': {'nombre_local': 'ATENAS (C)',
  'puntos_local': '69',
  'nombre_visita': 'BOCA',
  'puntos_visita': '81',
  'link_estadisticas': 'https://estadisticascabb.gesdeportiva.es/partido/rM2-eTJQHJsR2FLJYE8GRw==?a=1'},
 'OBRAS vs PLATENSE (008/10/2024 20:00)': {'nombre_local': 'OBRAS',
  'puntos_local': '94',
  'nombre_visita': 'PLATENSE',
  'puntos_visita': '90',
  'link_estadisticas': 'https://estadisticascabb.gesdeportiva.es/partido/UYiBwuNnKVL9qCbXhOY76g==?a=1'},
 'INSTITUTO vs OLIMPICO (LB) (008/10/2024 21:00)': {'nombre_local': 'INSTITUTO',
  'puntos_local': '99',
  'nombre_visita': 'OLIMPICO (LB)',
  'puntos_visita': '73',
  'link_estadisticas': 'https://estadisticascabb.gesdeportiva.es/partido/Dwgh5U3-8hNWcMtXCbn6yw==?a=1'},
 'UNION (SF) vs SAN LORENZO (008/10/2024 21:30)': {'nombre_local': 'UNION (SF)',
  'puntos_local': '68',
  'nombre_visita': 'SAN LORENZO',
  'puntos_visita': '63',
  'link_estadisticas': 'https://estadisticascabb.gesdepor

In [ ]:
match_data["ATENAS (C) vs BOCA (007/10/2024 22:10)"]

{'ATENAS (C) vs BOCA (007/10/2024 22:10)': {'nombre_local': 'ATENAS (C)',
  'puntos_local': '69',
  'nombre_visita': 'BOCA',
  'puntos_visita': '81',
  'link_estadisticas': 'https://estadisticascabb.gesdeportiva.es/partido/rM2-eTJQHJsR2FLJYE8GRw==?a=1'},
 'OBRAS vs PLATENSE (008/10/2024 20:00)': {'nombre_local': 'OBRAS',
  'puntos_local': '94',
  'nombre_visita': 'PLATENSE',
  'puntos_visita': '90',
  'link_estadisticas': 'https://estadisticascabb.gesdeportiva.es/partido/UYiBwuNnKVL9qCbXhOY76g==?a=1'},
 'INSTITUTO vs OLIMPICO (LB) (008/10/2024 21:00)': {'nombre_local': 'INSTITUTO',
  'puntos_local': '99',
  'nombre_visita': 'OLIMPICO (LB)',
  'puntos_visita': '73',
  'link_estadisticas': 'https://estadisticascabb.gesdeportiva.es/partido/Dwgh5U3-8hNWcMtXCbn6yw==?a=1'},
 'UNION (SF) vs SAN LORENZO (008/10/2024 21:30)': {'nombre_local': 'UNION (SF)',
  'puntos_local': '68',
  'nombre_visita': 'SAN LORENZO',
  'puntos_visita': '63',
  'link_estadisticas': 'https://estadisticascabb.gesdepor

Box score

In [ ]:
# --- Datos de ejemplo ---
# match_data = {
#     "ATENAS (C) vs BOCA (07/10/2024 22:10)": {
#         "link_estadisticas": "https://estadisticascabb.gesdeportiva.es/partido/rM2-eTJQHJsR2FLJYE8GRw==?a=1"
#     }
# }
# --- Fin del ejemplo ---

all_player_stats = []
driver = None

try:
    #primer_partido_key = list(match_data.keys())[0]  # Obtener el último partido del diccionario
    #primer_link = match_data[primer_partido_key]["link_estadisticas"]

    partido_key = 'GIMNASIA (CR) vs REGATAS (C) (003/05/2025 20:30)'
    partido_info = match_data[partido_key]["link_estadisticas"]
    print(f"El enlace del partido es: {partido_info}")

    # --- Parte 1: NAVEGACIÓN CON SELENIUM (YA FUNCIONA CORRECTAMENTE) ---
    print("\nIniciando Selenium...")
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get(partido_info)

    wait = WebDriverWait(driver, 20)

    print("Cambiando al iframe principal...")
    wait.until(EC.frame_to_be_available_and_switch_to_it((By.TAG_NAME, "iframe")))
    print("Cambio al iframe principal exitoso.")

    print("Buscando y haciendo clic en la pestaña 'Estadísticas'...")
    tab_selector = (By.CSS_SELECTOR, "li.pestana-estadisticas")
    estadisticas_tab_element = wait.until(EC.visibility_of_element_located(tab_selector))
    driver.execute_script("arguments[0].click();", estadisticas_tab_element)
    time.sleep(1)

    print("Cambiando al iframe de datos de estadísticas...")
    iframe_de_datos_selector = (By.CSS_SELECTOR, "div.contenido-estadisticas.activo iframe")
    wait.until(EC.frame_to_be_available_and_switch_to_it(iframe_de_datos_selector))
    print("Acceso al iframe de datos exitoso.")

    print("Extrayendo el código HTML final...")
    html_box_scores = driver.page_source

    # --- Parte 2: EXTRACCIÓN CON BEAUTIFULSOUP (SECCIÓN CORREGIDA) ---
    print("Procesando el HTML para extraer datos de jugadores...")
    soup = BeautifulSoup(html_box_scores, "html.parser")

    # --- INICIO DE LA CORRECCIÓN ---
    # Hacemos la búsqueda de los nombres de equipo de forma segura

    equipo_local_nombre = "Local" # Valor por defecto
    equipo_visitante_nombre = "Visitante" # Valor por defecto

    # Buscamos los divs que contienen los nombres de los equipos.
    # El selector 'div.nombre-equipo' es un candidato común.
    divs_nombres = soup.find_all("div", class_="nombre-equipo")
    if len(divs_nombres) >= 2:
        equipo_local_nombre = divs_nombres[0].get_text(strip=True)
        equipo_visitante_nombre = divs_nombres[1].get_text(strip=True)
    else:
        print("Advertencia: No se encontraron los nombres de equipo con el selector 'div.nombre-equipo'. Se usarán nombres genéricos.")
    # --- FIN DE LA CORRECCIÓN ---

    tablas = soup.find_all("table", class_="tabla-estadisticas")

    if len(tablas) >= 2:
        print(f"\nExtrayendo datos de: {equipo_local_nombre}")
        for fila in tablas[0].find("tbody").find_all("tr", onclick=True):
            onclick_attr = fila["onclick"]
            match = re.search(r"(\{.*\})", onclick_attr)
            if match:
                json_str = match.group(1).replace("'", '"')
                player_data = json.loads(json_str)
                player_data['equipo'] = equipo_local_nombre
                all_player_stats.append(player_data)

        print(f"Extrayendo datos de: {equipo_visitante_nombre}")
        for fila in tablas[1].find("tbody").find_all("tr", onclick=True):
            onclick_attr = fila["onclick"]
            match = re.search(r"(\{.*\})", onclick_attr)
            if match:
                json_str = match.group(1).replace("'", '"')
                player_data = json.loads(json_str)
                player_data['equipo'] = equipo_visitante_nombre
                all_player_stats.append(player_data)

    # --- Parte 3: PRESENTACIÓN DE DATOS CON PANDAS ---
    if all_player_stats:
        df_box_scores = pd.DataFrame(all_player_stats)
        print("\n✅ DataFrame final con todos los datos extraídos:")
        df_box_scores
    else:
        print("\nNo se pudieron extraer datos de jugadores del HTML.")

except Exception as e:
    print(f"Ocurrió un error general: {e}")
finally:
    if driver:
        driver.quit()
        print("\nNavegador cerrado.")

df_box_scores = df_box_scores[['IdJugador', 'IdClub', 'IdEquipo', 'Nombre', 'NombreCompleto',
                              'Puntos', 'TirosDos', 'TirosTres', 'TirosLibres',
                              'ReboteDefensivo', 'ReboteOfensivo', 'RebotesTotales',
                              'Asistencias', 'Recuperaciones', 'Perdidas',
                              'TaponCometido','TaponRecibido', 'FaltaCometida','FaltaRecibida','Valoracion',
                              'TiempoJuego', 'CincoInicial', 'equipo']]

# Lista de las columnas que contienen diccionarios para procesar
columnas_con_diccionarios = ['TirosDos', 'TirosTres', 'TirosLibres']

# Iteramos sobre cada columna que necesitamos transformar
for columna in columnas_con_diccionarios:
    if columna in df_box_scores.columns:
        # Creamos la nueva columna para 'Aciertos'
        # La función lambda toma cada diccionario (x) de la columna y extrae el valor de 'Aciertos'
        # .get('Aciertos', 0) es una forma segura que devuelve 0 si la clave 'Aciertos' no existe
        df_box_scores[f'{columna}Aciertos'] = df_box_scores[columna].apply(lambda x: x.get('Aciertos', 0))

        # Creamos la nueva columna para 'Fallos'
        df_box_scores[f'{columna}Fallos'] = df_box_scores[columna].apply(lambda x: x.get('Fallados', 0))

# Eliminamos las columnas originales que contenían los diccionarios para limpiar el DataFrame
df_box_scores = df_box_scores.drop(columns=columnas_con_diccionarios)
df_box_scores


El enlace del partido es: https://estadisticascabb.gesdeportiva.es/partido/EImi5Tc8raH75IyMO0gO6Q==?a=1

Iniciando Selenium...
Cambiando al iframe principal...
Cambio al iframe principal exitoso.
Buscando y haciendo clic en la pestaña 'Estadísticas'...
Cambiando al iframe de datos de estadísticas...
Acceso al iframe de datos exitoso.
Extrayendo el código HTML final...
Procesando el HTML para extraer datos de jugadores...

Extrayendo datos de: GIMNASIA (CR)
Extrayendo datos de: REGATAS (C)

✅ DataFrame final con todos los datos extraídos:

Navegador cerrado.


,IdJugador,IdClub,IdEquipo,Nombre,NombreCompleto,Puntos,ReboteDefensivo,ReboteOfensivo,RebotesTotales,Asistencias,Recuperaciones,Perdidas,TaponCometido,TaponRecibido,FaltaCometida,FaltaRecibida,Valoracion,TiempoJuego,CincoInicial,equipo,TirosDosAciertos,TirosDosFallos,TirosTresAciertos,TirosTresFallos,TirosLibresAciertos,TirosLibresFallos
0,219864,1474,68570,"CHACON, M.","CHACON TIRADO, MARCOS",11,2,0,2,2,0,0,0,0,4,2,9,16:59,False,GIMNASIA (CR),0,2,3,2,2,0
1,330064,1474,68570,"WHITFIELD, R.","WHITFIELD III, ROBERT JAMARCUS",21,2,0,2,1,0,0,0,0,1,0,22,13:32,False,GIMNASIA (CR),3,0,5,1,0,0
2,321289,1474,68570,"DE LOS SANTOS, N.","DE LOS SANTOS, NICOLAS",2,3,0,3,5,2,0,0,0,4,2,8,17:35,True,GIMNASIA (CR),1,2,0,0,0,0
3,321294,1474,68570,"CISNEROS, A.","CISNEROS SIERRA, ANYELO JOSE",13,1,2,3,4,3,0,0,0,2,0,20,21:55,False,GIMNASIA (CR),2,1,3,0,0,0
4,273711,1474,68570,"DATO, M.","DATO, MARTINIANO",9,3,0,3,4,2,2,0,0,1,1,14,17:58,True,GIMNASIA (CR),2,0,1,2,2,0
5,83757,1474,68570,"COPESKY, J.","COPESKY, JOSE IGNACIO",2,0,0,0,0,0,0,0,0,0,0,2,02:29,False,GIMNASIA (CR),1,0,0,0,0,0
6,269193,1474,68570,"TORETTA, E.","TORETTA, EMILIANO",9,2,1,3,4,0,2,0,0,2,1,8,19:56,False,GIMNASIA (CR),1,1,2,3,1,1
7,319448,1474,68570,"RIVERO, C.","RIVERO, CARLOS EZEQUIEL",6,1,1,2,0,2,0,0,0,3,0,6,17:40,False,GIMNASIA (CR),2,1,0,0,2,0
8,321287,1474,68570,"NATION, A.","NATION, ANDRE JAMES",16,3,0,3,3,1,2,0,0,1,3,18,26:28,True,GIMNASIA (CR),5,0,2,3,0,2
9,165056,1474,68570,"AYALA, V.","AYALA, VALENTINO",0,0,0,0,0,0,0,0,0,0,0,-2,02:29,False,GIMNASIA (CR),0,1,0,1,0,0


Play by Play


Obtener un partido

In [ ]:
acciones_del_partido = []
driver = None

try:
    # Asumiendo que 'match_data' ya fue creada
    partido_key = 'GIMNASIA (CR) vs REGATAS (C) (003/05/2025 20:30)'
    partido_link = match_data[partido_key]["link_estadisticas"]

    # --- Parte 1: NAVEGACIÓN CON SELENIUM ---
    print("\nIniciando Selenium...")
    options = Options()
    options.add_argument("--headless")  # Modo sin interfaz
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.binary_location = "/usr/bin/google-chrome"  # 🔹 le decimos explícitamente dónde está Chrome
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get(partido_link)

    wait = WebDriverWait(driver, 20)

    print("Cambiando al iframe principal...")
    wait.until(EC.frame_to_be_available_and_switch_to_it((By.TAG_NAME, "iframe")))
    print("Cambio al iframe principal exitoso.")

    print("Haciendo clic en la pestaña 'En vivo'...")
    en_vivo_tab = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.pestana-en-vivo")))
    driver.execute_script("arguments[0].click();", en_vivo_tab)

    print("Cambiando al iframe anidado del Play-by-Play...")
    iframe_pbp_selector = (By.CSS_SELECTOR, "div.contenido-en-vivo div:nth-child(2) iframe")
    wait.until(EC.frame_to_be_available_and_switch_to_it(iframe_pbp_selector))
    print("Acceso al iframe final del Play-by-Play exitoso.")

    html_pbp = driver.page_source

    # --- Parte 2: EXTRACCIÓN CON BEAUTIFULSOUP (CORREGIDA) ---
    print("Procesando el HTML para extraer las acciones del partido...")
    soup = BeautifulSoup(html_pbp, "html.parser")

    contenedor_acciones = soup.find("ul", class_="listadoAccionesPartido")

    if contenedor_acciones:
        acciones = contenedor_acciones.find_all("li", class_="accion")
        print(f"Se encontraron {len(acciones)} acciones en el partido.")

        for accion in acciones:
            # Inicializamos todas las variables
            tipo_accion = jugador = cuarto = tiempo = ""
            puntos_local = None
            puntos_visita = None

            titulo_tag = accion.find("strong", class_="titulo")
            if titulo_tag:
                tipo_accion = titulo_tag.get_text(strip=True)

            spans_info = accion.find_all("span", class_="informacion")
            if len(spans_info) >= 2:
                jugador = spans_info[0].get_text(strip=True)
                tiempo_text = spans_info[1].get_text(strip=True)
                match = re.search(r"Cuarto\s*(\d+)\s*-\s*(\d{2}:\d{2}:\d{2})", tiempo_text)
                if match:
                    cuarto = match.group(1)
                    tiempo = match.group(2)

            # --- NUEVO: Extraer el marcador de la acción ---
            marcador_tag = accion.find("strong", class_="informacionAdicional")
            if marcador_tag:
                marcador_texto = marcador_tag.get_text(strip=True)
                partes_marcador = marcador_texto.split('-')
                if len(partes_marcador) == 2:
                    try:
                        puntos_local = int(partes_marcador[0].strip())
                        puntos_visita = int(partes_marcador[1].strip())
                    except ValueError:
                        pass # Si hay un error, se quedan como None

            acciones_del_partido.append({
                "cuarto": cuarto,
                "tiempo": tiempo,
                "accion": tipo_accion,
                "jugador": jugador,
                "puntos_local": puntos_local,
                "puntos_visita": puntos_visita
            })

    # --- Parte 3: Crear y mostrar el DataFrame ---
    if acciones_del_partido:
        df_acciones = pd.DataFrame(acciones_del_partido)
        print("\n✅ DataFrame con marcador en cada acción:")
        display(df_acciones.head(10))
    else:
        print("No se pudieron extraer datos del Play-by-Play.")

except Exception as e:
    print(f"Ocurrió un error general: {e}")
finally:
    if driver:
        driver.quit()
        print("\nNavegador cerrado.")


Iniciando Selenium...
Cambiando al iframe principal...
Cambio al iframe principal exitoso.
Haciendo clic en la pestaña 'En vivo'...
Cambiando al iframe anidado del Play-by-Play...
Acceso al iframe final del Play-by-Play exitoso.
Procesando el HTML para extraer las acciones del partido...
Se encontraron 491 acciones en el partido.

✅ DataFrame con marcador en cada acción:


,cuarto,tiempo,accion,jugador,puntos_local,puntos_visita
0,,,FINAL DEL PERIODO4,,NaN,NaN
1,4,00:00:16,ASISTENCIA,"LARRAZA, JUAN IGNACIO",NaN,NaN
2,4,00:00:16,CANASTA DE 2 PUNTOS,"FRANCHELA, TOBIAS",109.0,70.0
3,4,00:00:20,REBOTE DEFENSIVO#44,"ALEXANDER, QUINTIN IMMANUEL",NaN,NaN
4,4,00:00:21,TIRO DE 2 FALLADO,"AYALA, VALENTINO",NaN,NaN
5,4,00:00:35,ASISTENCIA,"FRANCHELA, TOBIAS",NaN,NaN
6,4,00:00:35,CANASTA DE 2 PUNTOS,"ALEXANDER, QUINTIN IMMANUEL",109.0,68.0
7,4,00:00:48,CANASTA DE 2 PUNTOS,"WHITFIELD III, ROBERT JAMARCUS",109.0,66.0
8,4,00:01:16,TIRO LIBRE ANOTADO,"ALEXANDER, QUINTIN IMMANUEL",107.0,66.0
9,4,00:01:16,TIRO LIBRE ANOTADO,"ALEXANDER, QUINTIN IMMANUEL",107.0,65.0



Navegador cerrado.


Agregar quintetos y equipos

In [ ]:
print("Iniciando el proceso completo para generar quintetos por equipo...")

try:
    # === PASO 0: Identificar equipos local y visitante ===
    print("Paso 0: Identificando equipos local y visitante...")
    partido_key = 'GIMNASIA (CR) vs REGATAS (C) (003/05/2025 20:30)'
    nombre_local, nombre_visitante = partido_key.split(" vs ")
    nombre_visitante = nombre_visitante.split(" (")[0]

    def limpiar_equipo(nombre):
       return (
         str(nombre)
         .replace("(", "")
         .replace(")", "")
         .strip()
         .split()[0]  # nos quedamos solo con el nombre base
       )

    nombre_local = limpiar_equipo(nombre_local)
    nombre_visitante = limpiar_equipo(nombre_visitante)
    print(f"Equipo local: {nombre_local} | Equipo visitante: {nombre_visitante}")

    # === PASO 1: Crear el mapa de jugadores desde el Box Score ===

    print("Paso 1: Creando mapa de jugadores desde 'df_box_scores'...")
    if 'df_box_scores' not in locals() or df_box_scores.empty:
        raise NameError("El DataFrame 'df_box_scores' no existe o está vacío.")

    df_box_scores["NombreCompleto_limpio"] = (
    df_box_scores["NombreCompleto"]
    .str.strip()          # elimina espacios adelante y atrás
    .str.replace(r"\s+", " ", regex=True)  # colapsa espacios dobles
      )

    df_box_scores["equipo_normalizado"] = (
    df_box_scores["equipo"]
    .str.replace(r"\s*\(.*\)?", "", regex=True)
    .str.strip()
      )

    player_to_team_map = pd.Series(
    df_box_scores["equipo_normalizado"].values,
    index=df_box_scores["NombreCompleto_limpio"]
      ).to_dict()

    # === PASO 2: Generar la columna temporal 'quinteto_en_cancha' ===
    print("Paso 2: Generando lista de 10 jugadores en cancha por acción...")
    df_cronologico = df_acciones.iloc[::-1].reset_index(drop=True)

    jugadores_en_cancha = set()
    lista_de_quintetos_mixtos = []
    cuarto_actual = None  # opcional, si querés seguir rastreando el número de cuarto

    for index, row in df_cronologico.iterrows():
        accion = row.get('accion', '')
        jugador = row.get('jugador', '')
        accion_upper = accion.upper() if isinstance(accion, str) else ''
        cuarto = row.get('cuarto', None)

        # 🔁 Reiniciar SOLO si la acción indica "FINAL DEL PERIODO"
        if "FINAL DEL PERIODO" in accion_upper:
            jugadores_en_cancha = set()
            cuarto_actual = cuarto  # opcional
            # print(f"🔁 Reinicio por FINAL DEL PERIODO en fila {index}")

        # 🔹 Actualizar quién entra y sale
        elif "ENTRA A PISTA" in accion_upper or "CAMBIO-ENTRA" in accion_upper:
            if pd.notna(jugador) and jugador != '':
                jugadores_en_cancha.add(jugador)
        elif "ABANDONA LA PISTA" in accion_upper or "CAMBIO-SALE" in accion_upper:
            if pd.notna(jugador) and jugador != '':
                jugadores_en_cancha.discard(jugador)

        lista_de_quintetos_mixtos.append(sorted(list(jugadores_en_cancha)))

    df_cronologico['quinteto_en_cancha'] = lista_de_quintetos_mixtos


    # === PASO 3: Separar el quinteto mixto en local y visitante ===
    print("Paso 3: Separando la lista de 10 jugadores en quintetos por equipo...")

    def dividir_quinteto(quinteto_mixto, roster_map, equipo_local_nombre):
        quinteto_local, quinteto_visita = [], []
        for jugador in quinteto_mixto:
            jugador_limpio = (
                str(jugador).strip()
                .replace("  ", " ")
            )
            equipo = roster_map.get(jugador_limpio)
            if equipo == equipo_local_nombre:
                quinteto_local.append(jugador_limpio)
            else:
                quinteto_visita.append(jugador_limpio)
        return sorted(quinteto_local), sorted(quinteto_visita)

    nuevas_columnas = df_cronologico['quinteto_en_cancha'].apply(
        lambda q: pd.Series(dividir_quinteto(q, player_to_team_map, nombre_local))
    )
    nuevas_columnas.columns = ['quinteto_local', 'quinteto_visita']

    df_cronologico = pd.concat([df_cronologico, nuevas_columnas], axis=1)

    df_cronologico['jugadores_en_cancha'] = df_cronologico['quinteto_en_cancha'].apply(
    lambda x: len(x) if isinstance(x, (list, tuple)) else 0
    )

    # === PASO 4: Agregar columnas de equipo local y visitante ===
    df_cronologico["equipo_local"] = nombre_local
    df_cronologico["equipo_visitante"] = nombre_visitante

    # === PASO 4B: Identificar a qué equipo corresponde cada acción ===
    print("Paso 4B: Identificando equipo responsable de cada acción...")

    df_cronologico["jugador_limpio"] = (
        df_cronologico["jugador"]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    def identificar_equipo(row):
        jugador = row.get("jugador_limpio")
        if pd.notna(jugador) and jugador in player_to_team_map:
            return player_to_team_map[jugador]
        accion = row.get("accion", "").upper()
        if "LOCAL" in accion:
            return nombre_local
        elif "VISITANTE" in accion:
            return nombre_visitante
        return None

    df_cronologico["equipo_accion"] = df_cronologico.apply(identificar_equipo, axis=1)
    df_cronologico["tipo_equipo_accion"] = df_cronologico["equipo_accion"].apply(
        lambda eq: "local" if eq == nombre_local else ("visitante" if eq == nombre_visitante else None)
    )

    # === PASO 5: Finalizar ===
    df_cronologico = df_cronologico.drop(columns=['quinteto_en_cancha'])
    df_acciones_final = df_cronologico.iloc[::-1].reset_index(drop=True)

    print("\n✅ ¡Proceso completado! DataFrame final generado.")
    display(df_acciones_final.tail(10))

except Exception as e:
    print(f"❌ Ocurrió un error en el script: {e}")

Iniciando el proceso completo para generar quintetos por equipo...
Paso 0: Identificando equipos local y visitante...
Equipo local: GIMNASIA | Equipo visitante: REGATAS
Paso 1: Creando mapa de jugadores desde 'df_box_scores'...
Paso 2: Generando lista de 10 jugadores en cancha por acción...
Paso 3: Separando la lista de 10 jugadores en quintetos por equipo...
Paso 4B: Identificando equipo responsable de cada acción...

✅ ¡Proceso completado! DataFrame final generado.


,cuarto,tiempo,accion,jugador,puntos_local,puntos_visita,quinteto_local,quinteto_visita,jugadores_en_cancha,equipo_local,equipo_visitante,jugador_limpio,equipo_accion,tipo_equipo_accion
481,1,00:10:00,ENTRA A PISTA EL#33,"RAMIREZ BARRIOS, CLAUDIO FABIAN",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, GRAMAJO, PABLO IVAN...",9,GIMNASIA,REGATAS,"RAMIREZ BARRIOS, CLAUDIO FABIAN",REGATAS,visitante
482,1,00:10:00,ENTRA A PISTA EL#21,"THOMAS III, CHARLES PRICE",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, GRAMAJO, PABLO IVAN...",8,GIMNASIA,REGATAS,"THOMAS III, CHARLES PRICE",REGATAS,visitante
483,1,00:10:00,ENTRA A PISTA EL#7,"AGUIRRE, GUSTAVO NICOLAS",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, GRAMAJO, PABLO IVAN]",7,GIMNASIA,REGATAS,"AGUIRRE, GUSTAVO NICOLAS",REGATAS,visitante
484,1,00:10:00,ENTRA A PISTA EL#5,"GRAMAJO, PABLO IVAN",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[GRAMAJO, PABLO IVAN]",6,GIMNASIA,REGATAS,"GRAMAJO, PABLO IVAN",REGATAS,visitante
485,1,00:10:00,ENTRA A PISTA EL#30,"HORTON, KENNETH WILCHER",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...",[],5,GIMNASIA,REGATAS,"HORTON, KENNETH WILCHER",GIMNASIA,local
486,1,00:10:00,ENTRA A PISTA EL#23,"DANIELS, TRAVIS DORREL",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...",[],4,GIMNASIA,REGATAS,"DANIELS, TRAVIS DORREL",GIMNASIA,local
487,1,00:10:00,ENTRA A PISTA EL#15,"NATION, ANDRE JAMES",NaN,NaN,"[DATO, MARTINIANO, DE LOS SANTOS, NICOLAS, NAT...",[],3,GIMNASIA,REGATAS,"NATION, ANDRE JAMES",GIMNASIA,local
488,1,00:10:00,ENTRA A PISTA EL#9,"DATO, MARTINIANO",NaN,NaN,"[DATO, MARTINIANO, DE LOS SANTOS, NICOLAS]",[],2,GIMNASIA,REGATAS,"DATO, MARTINIANO",GIMNASIA,local
489,1,00:10:00,ENTRA A PISTA EL#5,"DE LOS SANTOS, NICOLAS",NaN,NaN,"[DE LOS SANTOS, NICOLAS]",[],1,GIMNASIA,REGATAS,"DE LOS SANTOS, NICOLAS",GIMNASIA,local
490,,,INICIO DEL PARTIDO,,NaN,NaN,[],[],0,GIMNASIA,REGATAS,,None,None


In [ ]:
#df_acciones_final.to_excel('acciones_partido_crudo_prueba.xlsx', index=False)
#print("DataFrame 'df_acciones_final' guardado como 'acciones_partido.xlsx'")


### 1 partido - Preprocesado

In [ ]:
import unicodedata
import re

def quitar_tildes(texto):
    if isinstance(texto, str):
        return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

def normalizar_accion_simple(texto):
    s = str(texto).upper()
    s = quitar_tildes(s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# copia de trabajo
df_preprocesada = df_acciones_final.copy()

# 1) Filtrar por jugadores en cancha
df_preprocesada = df_preprocesada[df_preprocesada["jugadores_en_cancha"] >= 10]

# 2) Normalizar acción en una sola columna que usamos para todo
df_preprocesada['accion_norm'] = df_preprocesada['accion'].apply(normalizar_accion_simple)

# 3) Eliminar acciones prohibidas (usamos accion_norm)
strings_a_eliminar = [
    "ENTRA A PISTA",
    "INICIO DEL PERIODO",
    "FINAL DEL PERIODO",
    "ABANDONA LA PISTA",
    "TAPON REALIZADO",
    "TECNICA",
    "TECNICA BANQUILLO (C)",
    "TECNICA BANQUILLO (B)",
    "TECNICA DESCALIFICANTE",
    "TECNICA DESCALIFICANTE ACOMPAÑANTE NO PARTICIPA",
    "TECNICA-E (ALINEACIÓN ILEGAL)",
    "TIEMPO MUERTO SOLICITADO",
    "ANTIDEPORTIVA"
]
escaped_strings = [re.escape(s) for s in strings_a_eliminar]
patron = "|".join(escaped_strings)
df_preprocesada = df_preprocesada[~df_preprocesada['accion_norm'].str.contains(patron, regex=True, na=False)].copy()

# 4) Detectar FLECHA DE ALTERNANCIA y voltear equipos
mask_flecha = df_preprocesada['accion_norm'].str.contains('FLECHA ALTERNANCIA', na=False)
df_preprocesada.loc[mask_flecha, "equipo_accion"] = df_preprocesada.loc[mask_flecha, "equipo_accion"].map({
    nombre_local: nombre_visitante,
    nombre_visitante: nombre_local
})
df_preprocesada.loc[mask_flecha, "tipo_equipo_accion"] = df_preprocesada.loc[mask_flecha, "tipo_equipo_accion"].map({
    "local": "visitante",
    "visitante": "local"
})

#5)

mask_last_in_quarter = df_preprocesada['cuarto'].fillna(-999) != df_preprocesada['cuarto'].shift(1).fillna(-999)
mask_rebote_def = df_preprocesada['accion_norm'].str.contains(r'\bREBOTE DEFENSIVO\b', regex=True, na=False)
mask_remove_last_rebote = mask_last_in_quarter & mask_rebote_def

n_removed_rebotes = int(mask_remove_last_rebote.sum())
if n_removed_rebotes:
    df_preprocesada = df_preprocesada.loc[~mask_remove_last_rebote].copy()
print(f"Eliminadas {n_removed_rebotes} filas (últimos acciones de cuarto que eran REBOTE DEFENSIVO).")

#6)
# --- función para convertir HH:MM:SS a segundos ---
def tiempo_a_segundos(t):
    try:
        hh, mm, ss = t.split(":")
        return int(hh) * 3600 + int(mm) * 60 + int(ss)
    except:
        return float('nan')

# acción previa cronológica (shift(-1))
accion_prev = df_preprocesada['accion_norm'].shift(-1).fillna('')
equipo_prev = df_preprocesada['equipo_accion'].shift(-1)

# máscaras previas
mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)?\b', regex=True, na=False)
mask_flecha_prev = accion_prev.str.contains('FLECHA ALTERNANCIA', na=False)
mask_perdida_prev = accion_prev.str.contains(r'\bPERDIDA\b', regex=True, na=False)
mask_asistencia_prev = accion_prev.str.contains(r'\bASISTENCIA\b', regex=True, na=False)

# scoring previo (TRIPLE o CANASTA 2)
mask_triple_prev = accion_prev.str.contains(r'\bTRIPLE\b', na=False)
mask_canasta2_prev = accion_prev.str.contains(r'\bCANASTA\b', na=False) & accion_prev.str.contains(r'\b2\b', na=False)

# --- diferencia de tiempo ---
tiempo_prev_seg = df_preprocesada['tiempo'].shift(-1).apply(tiempo_a_segundos)
tiempo_curr_seg = df_preprocesada['tiempo'].apply(tiempo_a_segundos)

elapsed = tiempo_prev_seg - tiempo_curr_seg   # el reloj baja → positivo si pasó tiempo real

# excepción por scoring + >4 segundos
mask_scoring_prev_diff_time = (
    (mask_triple_prev | mask_canasta2_prev | mask_asistencia_prev)
    & (elapsed > 4)
)

mask_prev_excepcion = (
    mask_tiro_prev
    | mask_flecha_prev
    | mask_perdida_prev
    | mask_scoring_prev_diff_time
)

# faltas
mask_falta = df_preprocesada['accion_norm'] == 'FALTA COMETIDA'

# equipo distinto
mask_equipo_distinto = df_preprocesada['equipo_accion'] != equipo_prev

# mantener falta si cumple excepción + equipo distinto
mask_keep_falta = mask_falta & mask_prev_excepcion & mask_equipo_distinto

# eliminar el resto
mask_a_eliminar = mask_falta & (~mask_keep_falta)
df_preprocesada = df_preprocesada.loc[~mask_a_eliminar].copy()

# 6) limpieza final
df_preprocesada = df_preprocesada.drop(columns=['accion_norm'])

#7) agrego ultimas columnas
df_tmp = df_preprocesada.iloc[::-1].reset_index(drop=True)

# -------------------------------------------------------------------
# 1) posesiones: sube cuando cambia equipo_accion o cambia cuarto
# -------------------------------------------------------------------

cond_nueva_posesion = (
    (df_tmp['equipo_accion'] != df_tmp['equipo_accion'].shift(1)) |
    (df_tmp['cuarto'] != df_tmp['cuarto'].shift(1))
)
cond_nueva_posesion.iloc[0] = True  # primera acción = posesión nueva
df_tmp['posesion'] = cond_nueva_posesion.cumsum().astype(int)

# -------------------------------------------------------------------
# 2) quintetos_por_pos: pares (quinteto_local, quinteto_visita)
# -------------------------------------------------------------------

def contar_quintetos(gr):
    pares = list(zip(gr['quinteto_local'].fillna('__NA__'),
                     gr['quinteto_visita'].fillna('__NA__')))
    return len(pd.Series(pares).drop_duplicates())

quintetos_por_pos = df_tmp.groupby('posesion').apply(contar_quintetos)
quintetos_por_pos.name = 'quintetos_por_pos'

df_tmp = df_tmp.join(quintetos_por_pos, on='posesion')

# -------------------------------------------------------------------
# 3) / 4) pts_acum_local & pts_acum_visit
# Tus columnas YA traen el marcador actualizado cuando hay puntos.
# Entonces sólo hacemos forward-fill y rellenamos con 0 al inicio.
# -------------------------------------------------------------------

df_tmp['pts_acum_local'] = df_tmp['puntos_local'].ffill().fillna(0).astype(int)
df_tmp['pts_acum_visit'] = df_tmp['puntos_visita'].ffill().fillna(0).astype(int)

# -------------------------------------------------------------------
# 5) pts_fin_loc / pts_fin_vis
# Solo en la última fila de cada posesión (en orden cronológico real).
# -------------------------------------------------------------------

es_ultima_fila_posesion = df_tmp['posesion'] != df_tmp['posesion'].shift(-1)

df_tmp['pts_fin_loc'] = None
df_tmp['pts_fin_vis'] = None

df_tmp.loc[es_ultima_fila_posesion, 'pts_fin_loc'] = df_tmp.loc[es_ultima_fila_posesion, 'pts_acum_local']
df_tmp.loc[es_ultima_fila_posesion, 'pts_fin_vis'] = df_tmp.loc[es_ultima_fila_posesion, 'pts_acum_visit']

# -------------------------------------------------------------------
# 6) VOLVER AL ORDEN ORIGINAL de tu base
# (opcional, pero lo pediste)
# -------------------------------------------------------------------

df_preprocesada = df_tmp.iloc[::-1].reset_index(drop=True)

display(df_preprocesada.head(10))
print(f"Eliminadas {mask_a_eliminar.sum()} filas 'FALTA COMETIDA' que no cumplían la excepción.")
df_preprocesada.to_excel('acciones_partido_preprocesado.xlsx', index=False)


Eliminadas 1 filas (últimos acciones de cuarto que eran REBOTE DEFENSIVO).


/tmp/ipython-input-854659341.py:76: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)?\b', regex=True, na=False)
/tmp/ipython-input-854659341.py:143: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  quintetos_por_pos = df_tmp.groupby('posesion').apply(contar_quintetos)


,cuarto,tiempo,accion,jugador,puntos_local,puntos_visita,quinteto_local,quinteto_visita,jugadores_en_cancha,equipo_local,equipo_visitante,jugador_limpio,equipo_accion,tipo_equipo_accion,posesion,quintetos_por_pos,pts_acum_local,pts_acum_visit,pts_fin_loc,pts_fin_vis
0,4,00:00:16,ASISTENCIA,"LARRAZA, JUAN IGNACIO",NaN,NaN,"[AYALA, VALENTINO, CISNEROS SIERRA, ANYELO JOS...","[ALEXANDER, QUINTIN IMMANUEL, FRANCHELA, TOBIA...",10,GIMNASIA,REGATAS,"LARRAZA, JUAN IGNACIO",REGATAS,visitante,148,1,109,70,109,70
1,4,00:00:16,CANASTA DE 2 PUNTOS,"FRANCHELA, TOBIAS",109.0,70.0,"[AYALA, VALENTINO, CISNEROS SIERRA, ANYELO JOS...","[ALEXANDER, QUINTIN IMMANUEL, FRANCHELA, TOBIA...",10,GIMNASIA,REGATAS,"FRANCHELA, TOBIAS",REGATAS,visitante,148,1,109,70,None,None
2,4,00:00:20,REBOTE DEFENSIVO#44,"ALEXANDER, QUINTIN IMMANUEL",NaN,NaN,"[AYALA, VALENTINO, CISNEROS SIERRA, ANYELO JOS...","[ALEXANDER, QUINTIN IMMANUEL, FRANCHELA, TOBIA...",10,GIMNASIA,REGATAS,"ALEXANDER, QUINTIN IMMANUEL",REGATAS,visitante,148,1,109,68,None,None
3,4,00:00:21,TIRO DE 2 FALLADO,"AYALA, VALENTINO",NaN,NaN,"[AYALA, VALENTINO, CISNEROS SIERRA, ANYELO JOS...","[ALEXANDER, QUINTIN IMMANUEL, FRANCHELA, TOBIA...",10,GIMNASIA,REGATAS,"AYALA, VALENTINO",GIMNASIA,local,147,1,109,68,109,68
4,4,00:00:35,ASISTENCIA,"FRANCHELA, TOBIAS",NaN,NaN,"[AYALA, VALENTINO, CISNEROS SIERRA, ANYELO JOS...","[ALEXANDER, QUINTIN IMMANUEL, FRANCHELA, TOBIA...",10,GIMNASIA,REGATAS,"FRANCHELA, TOBIAS",REGATAS,visitante,146,1,109,68,109,68
5,4,00:00:35,CANASTA DE 2 PUNTOS,"ALEXANDER, QUINTIN IMMANUEL",109.0,68.0,"[AYALA, VALENTINO, CISNEROS SIERRA, ANYELO JOS...","[ALEXANDER, QUINTIN IMMANUEL, FRANCHELA, TOBIA...",10,GIMNASIA,REGATAS,"ALEXANDER, QUINTIN IMMANUEL",REGATAS,visitante,146,1,109,68,None,None
6,4,00:00:48,CANASTA DE 2 PUNTOS,"WHITFIELD III, ROBERT JAMARCUS",109.0,66.0,"[AYALA, VALENTINO, CISNEROS SIERRA, ANYELO JOS...","[ALEXANDER, QUINTIN IMMANUEL, FRANCHELA, TOBIA...",10,GIMNASIA,REGATAS,"WHITFIELD III, ROBERT JAMARCUS",GIMNASIA,local,145,1,109,66,109,66
7,4,00:01:16,TIRO LIBRE ANOTADO,"ALEXANDER, QUINTIN IMMANUEL",107.0,66.0,"[AYALA, VALENTINO, CISNEROS SIERRA, ANYELO JOS...","[ALEXANDER, QUINTIN IMMANUEL, FRANCHELA, TOBIA...",10,GIMNASIA,REGATAS,"ALEXANDER, QUINTIN IMMANUEL",REGATAS,visitante,144,1,107,66,107,66
8,4,00:01:16,TIRO LIBRE ANOTADO,"ALEXANDER, QUINTIN IMMANUEL",107.0,65.0,"[AYALA, VALENTINO, CISNEROS SIERRA, ANYELO JOS...","[ALEXANDER, QUINTIN IMMANUEL, FRANCHELA, TOBIA...",10,GIMNASIA,REGATAS,"ALEXANDER, QUINTIN IMMANUEL",REGATAS,visitante,144,1,107,65,None,None
9,4,00:01:16,2 TIROS LIBRES PARA EL#44,"ALEXANDER, QUINTIN IMMANUEL",NaN,NaN,"[AYALA, VALENTINO, CISNEROS SIERRA, ANYELO JOS...","[ALEXANDER, QUINTIN IMMANUEL, FRANCHELA, TOBIA...",10,GIMNASIA,REGATAS,"ALEXANDER, QUINTIN IMMANUEL",REGATAS,visitante,144,1,107,64,None,None


Eliminadas 32 filas 'FALTA COMETIDA' que no cumplían la excepción.


Base poss - by - poss


In [ ]:
# ---------------------------
# CREAR poss-by-poss (última play de cada posesión) + indicadores
# puntos_pos = diferencia entre pts_fin de esta posesión y pts_fin de la posesión anterior
# ---------------------------

import ast
import numpy as np
import pandas as pd

# 0) FILTRAR filas que son finales de posesión (última play de cada posesión)
df_poss_rows = df_preprocesada[
    df_preprocesada['pts_fin_loc'].notna() | df_preprocesada['pts_fin_vis'].notna()
].copy().reset_index(drop=True)

# 1) Parser robusto para quintetos (lista real, np.array, string "['A','B']", etc.)
def parse_quinteto_python_list(x):
    # None / NaN (escalar)
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    # listas/tuplas/sets/ndarray
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return [str(i).strip() for i in list(x) if pd.notna(i) and str(i).strip()!='']
    # string
    s = str(x).strip()
    if s.lower() in ('nan', 'none', ''):
        return []
    # intentar literal_eval si parece lista
    if s.startswith('[') and s.endswith(']'):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set, np.ndarray)):
                return [str(i).strip() for i in list(parsed) if pd.notna(i) and str(i).strip()!='']
        except Exception:
            pass
    # fallback: devolver el string completo como un elemento
    return [s]

# 2) Construir lista completa de jugadores (desde 'jugador' y desde los quintetos)
jugadores_desde_col = df_preprocesada['jugador'].dropna().astype(str).unique().tolist()

todos_q = []
for q in pd.concat([df_preprocesada['quinteto_local'].fillna('[]'),
                    df_preprocesada['quinteto_visita'].fillna('[]')], ignore_index=True):
    parsed = parse_quinteto_python_list(q)
    todos_q.extend(parsed)

# lista final de jugadores (ordenada)
jugadores = sorted(set(jugadores_desde_col + todos_q))

# 3) Asegurar numericidad en pts_fin (en la tabla de filas de posesiones)
for c in ['pts_fin_loc','pts_fin_vis','pts_acum_local','pts_acum_visit']:
    if c in df_poss_rows.columns:
        df_poss_rows[c] = pd.to_numeric(df_poss_rows[c], errors='coerce')

# 4) Parsear quintetos en listas para cada fila (columnas temporales)
df_poss_rows['q_local_list'] = df_poss_rows['quinteto_local'].apply(parse_quinteto_python_list)
df_poss_rows['q_vis_list']   = df_poss_rows['quinteto_visita'].apply(parse_quinteto_python_list)

# 5) Ordenar por posesion ascendente (asegura que shift(1) sea la posesión anterior)
df_poss_rows = df_poss_rows.sort_values('posesion').reset_index(drop=True)

# 6) Determinar si en cada posesión atacaba el local (vectorizado, preferimos tipo_equipo_accion, sino comparación nombres)
def ataca_local_row(r):
    tipo = r.get('tipo_equipo_accion')
    equipo_acc = r.get('equipo_accion')
    equipo_local = r.get('equipo_local')
    if isinstance(tipo, str) and tipo.strip() != '':
        t = tipo.lower()
        if 'local' in t:
            return True
        if 'visit' in t:
            return False
    # fallback a comparación por nombre
    return (pd.notna(equipo_acc) and pd.notna(equipo_local) and str(equipo_acc) == str(equipo_local))

df_poss_rows['ataca_local'] = df_poss_rows.apply(ataca_local_row, axis=1)

# 7) Preparar columnas pts_fin para cálculo (reemplazar NaN por 0 para evitar errores en la resta;
# pero la lógica de diferencia es con la misma columna anterior)
df_poss_rows['pts_fin_loc_f'] = df_poss_rows['pts_fin_loc'].fillna(0).astype(int)
df_poss_rows['pts_fin_vis_f'] = df_poss_rows['pts_fin_vis'].fillna(0).astype(int)

# 8) Calcular puntos por posesión usando diferencia con la posesión anterior (shift)
# Para la primera fila, shift dará NaN -> lo rellenamos por 0 (equivale a comparar con 0)
prev_pts_fin_loc = df_poss_rows['pts_fin_loc_f'].shift(1).fillna(0).astype(int)
prev_pts_fin_vis = df_poss_rows['pts_fin_vis_f'].shift(1).fillna(0).astype(int)

# aplicar la regla: si ataca local -> pts_fin_loc - prev_pts_fin_loc, else pts_fin_vis - prev_pts_fin_vis
df_poss_rows['puntos_pos'] = np.where(
    df_poss_rows['ataca_local'],
    (df_poss_rows['pts_fin_loc_f'] - prev_pts_fin_loc).astype(int),
    (df_poss_rows['pts_fin_vis_f'] - prev_pts_fin_vis).astype(int)
).astype(int)

# 9) Crear indicadores por jugador (1 ataque, -1 defensa, 0 no en cancha)
def obtener_atac_def(row):
    q_local = set(row['q_local_list'])
    q_vis = set(row['q_vis_list'])
    equipo_acc = row.get('equipo_accion')
    equipo_local = row.get('equipo_local')
    equipo_visit = row.get('equipo_visitante')
    tipo_equipo_acc = row.get('tipo_equipo_accion', None)

    # priorizar comparación explícita
    if pd.notna(equipo_acc) and pd.notna(equipo_local) and str(equipo_acc) == str(equipo_local):
        return q_local, q_vis
    if pd.notna(equipo_acc) and pd.notna(equipo_visit) and str(equipo_acc) == str(equipo_visit):
        return q_vis, q_local

    # fallback a tipo_equipo_accion
    if isinstance(tipo_equipo_acc, str):
        t = tipo_equipo_acc.lower()
        if 'local' in t:
            return q_local, q_vis
        if 'visit' in t:
            return q_vis, q_local

    # último recurso: local ataca
    return q_local, q_vis

indic_rows = []
for _, row in df_poss_rows.iterrows():
    atac, defen = obtener_atac_def(row)
    indic = {p: 0 for p in jugadores}
    for p in atac:
        if p in indic:
            indic[p] = 1
    for p in defen:
        if p in indic:
            indic[p] = -1
    indic_rows.append(indic)

df_indicadores = pd.DataFrame(indic_rows, columns=jugadores)

# 10) Concatenar: conservar todas las columnas originales de df_poss_rows + puntos_pos + indicadores
# (df_poss_rows ya incluye las columnas temporales; vamos a quedarnos con las originales + puntos_pos + ataca_local y luego los jugadores)
cols_originales = [c for c in df_poss_rows.columns if c not in ('q_local_list','q_vis_list','pts_fin_loc_f','pts_fin_vis_f')]
# aseguramos que 'puntos_pos' y 'ataca_local' estén en cols_originales (si no, los añadimos)
if 'puntos_pos' not in cols_originales:
    cols_originales = cols_originales + ['puntos_pos']
if 'ataca_local' not in cols_originales:
    cols_originales = cols_originales + ['ataca_local']

df_poss_by_poss = pd.concat([df_poss_rows[cols_originales].reset_index(drop=True), df_indicadores.reset_index(drop=True)], axis=1)

# 11) Orden final: por posesion asc (opcional)
if 'posesion' in df_poss_by_poss.columns:
    df_poss_by_poss = df_poss_by_poss.sort_values('posesion').reset_index(drop=True)

# 12) Mostrar resultado
display(df_poss_by_poss.head(30))

# Opcional: guardar
df_poss_by_poss.to_excel('poss_by_poss.xlsx', index=False)
# df_poss_by_poss.to_csv('poss_by_poss_with_indicators_and_puntos_pos.csv', index=False)


,cuarto,tiempo,accion,jugador,puntos_local,puntos_visita,quinteto_local,quinteto_visita,jugadores_en_cancha,equipo_local,equipo_visitante,jugador_limpio,equipo_accion,tipo_equipo_accion,posesion,quintetos_por_pos,pts_acum_local,pts_acum_visit,pts_fin_loc,pts_fin_vis,ataca_local,puntos_pos,,"AGUIRRE, GUSTAVO NICOLAS","ALEXANDER, QUINTIN IMMANUEL","AYALA, VALENTINO","CHACON TIRADO, MARCOS","CISNEROS SIERRA, ANYELO JOSE","COPESKY, JOSE IGNACIO","DANIELS, TRAVIS DORREL","DATO, MARTINIANO","DE LOS SANTOS, NICOLAS","FRANCHELA, TOBIAS","GILETTO, SALVADOR","GRAMAJO, PABLO IVAN","HORTON, KENNETH WILCHER","JAIME, ANDRES JOAQUIN","LARRAZA, JUAN IGNACIO","LUGO, SEBASTIAN EZEQUIEL","NATION, ANDRE JAMES","RAMIREZ BARRIOS, CLAUDIO FABIAN","RIVERO, CARLOS EZEQUIEL","THOMAS III, CHARLES PRICE","TORETTA, EMILIANO","WHITFIELD III, ROBERT JAMARCUS"
0,1,00:09:47,ASISTENCIA,"DE LOS SANTOS, NICOLAS",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, FRANCHELA, TOBIAS, ...",10,GIMNASIA,REGATAS,"DE LOS SANTOS, NICOLAS",GIMNASIA,local,1,1,2,0,2,0,True,2,0,-1,0,0,0,0,0,1,1,1,-1,0,-1,1,0,0,0,1,-1,0,-1,0,0
1,1,00:09:24,CANASTA DE 2 PUNTOS,"GRAMAJO, PABLO IVAN",2.0,4.0,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, FRANCHELA, TOBIAS, ...",10,GIMNASIA,REGATAS,"GRAMAJO, PABLO IVAN",REGATAS,visitante,2,1,2,4,2,4,False,4,0,1,0,0,0,0,0,-1,-1,-1,1,0,1,-1,0,0,0,-1,1,0,1,0,0
2,1,00:09:09,CANASTA DE 2 PUNTOS,"DATO, MARTINIANO",4.0,4.0,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, FRANCHELA, TOBIAS, ...",10,GIMNASIA,REGATAS,"DATO, MARTINIANO",GIMNASIA,local,3,1,4,4,4,4,True,2,0,-1,0,0,0,0,0,1,1,1,-1,0,-1,1,0,0,0,1,-1,0,-1,0,0
3,1,00:09:01,CANASTA DE 2 PUNTOS,"THOMAS III, CHARLES PRICE",4.0,6.0,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, FRANCHELA, TOBIAS, ...",10,GIMNASIA,REGATAS,"THOMAS III, CHARLES PRICE",REGATAS,visitante,4,1,4,6,4,6,False,2,0,1,0,0,0,0,0,-1,-1,-1,1,0,1,-1,0,0,0,-1,1,0,1,0,0
4,1,00:08:40,PÉRDIDA DE BALÓN,"DANIELS, TRAVIS DORREL",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, FRANCHELA, TOBIAS, ...",10,GIMNASIA,REGATAS,"DANIELS, TRAVIS DORREL",GIMNASIA,local,5,1,4,6,4,6,True,0,0,-1,0,0,0,0,0,1,1,1,-1,0,-1,1,0,0,0,1,-1,0,-1,0,0
5,1,00:08:33,CANASTA DE 2 PUNTOS,"FRANCHELA, TOBIAS",4.0,8.0,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, FRANCHELA, TOBIAS, ...",10,GIMNASIA,REGATAS,"FRANCHELA, TOBIAS",REGATAS,visitante,6,1,4,8,4,8,False,2,0,1,0,0,0,0,0,-1,-1,-1,1,0,1,-1,0,0,0,-1,1,0,1,0,0
6,1,00:08:19,ASISTENCIA,"DE LOS SANTOS, NICOLAS",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, FRANCHELA, TOBIAS, ...",10,GIMNASIA,REGATAS,"DE LOS SANTOS, NICOLAS",GIMNASIA,local,7,1,6,8,6,8,True,2,0,-1,0,0,0,0,0,1,1,1,-1,0,-1,1,0,0,0,1,-1,0,-1,0,0
7,1,00:08:00,TIRO DE 3 FALLADO,"RAMIREZ BARRIOS, CLAUDIO FABIAN",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, FRANCHELA, TOBIAS, ...",10,GIMNASIA,REGATAS,"RAMIREZ BARRIOS, CLAUDIO FABIAN",REGATAS,visitante,8,1,6,8,6,8,False,0,0,1,0,0,0,0,0,-1,-1,-1,1,0,1,-1,0,0,0,-1,1,0,1,0,0
8,1,00:07:51,TIRO DE 3 FALLADO,"HORTON, KENNETH WILCHER",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, FRANCHELA, TOBIAS, ...",10,GIMNASIA,REGATAS,"HORTON, KENNETH WILCHER",GIMNASIA,local,9,1,6,8,6,8,True,0,0,-1,0,0,0,0,0,1,1,1,-1,0,-1,1,0,0,0,1,-1,0,-1,0,0
9,1,00:07:28,TIRO DE 3 FALLADO,"RAMIREZ BARRIOS, CLAUDIO FABIAN",NaN,NaN,"[DANIELS, TRAVIS DORREL, DATO, MARTINIANO, DE ...","[AGUIRRE, GUSTAVO NICOLAS, FRANCHELA, TOBIAS, ...",10,GIMNASIA,REGATAS,"RAMIREZ BARRIOS, CLAUDIO FABIAN",REGATAS,visitante,10,1,6,8,6,8,False,0,0,1,0,0,0,0,0,-1,-1,-1,1,0,1,-1,0,0,0,-1,1,0,1,0,0


### 10 partidos - crudos

In [ ]:
def extraer_boxscore_y_pbp(partido_link):
    """Extrae boxscore y play-by-play de un partido dado su link."""
    all_player_stats = []
    acciones_del_partido = []
    driver = None
    df_box_scores = None
    df_acciones_final = None

    try:
        # --- BOX SCORE ---
        options = Options()
        options.add_argument("--headless")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        driver.get(partido_link)
        wait = WebDriverWait(driver, 20)
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.TAG_NAME, "iframe")))
        tab_selector = (By.CSS_SELECTOR, "li.pestana-estadisticas")
        estadisticas_tab_element = wait.until(EC.visibility_of_element_located(tab_selector))
        driver.execute_script("arguments[0].click();", estadisticas_tab_element)
        time.sleep(1)
        iframe_de_datos_selector = (By.CSS_SELECTOR, "div.contenido-estadisticas.activo iframe")
        wait.until(EC.frame_to_be_available_and_switch_to_it(iframe_de_datos_selector))
        html_box_scores = driver.page_source
        soup = BeautifulSoup(html_box_scores, "html.parser")
        divs_nombres = soup.find_all("div", class_="nombre-equipo")
        equipo_local_nombre = divs_nombres[0].get_text(strip=True) if len(divs_nombres) >= 2 else "Local"
        equipo_visitante_nombre = divs_nombres[1].get_text(strip=True) if len(divs_nombres) >= 2 else "Visitante"
        tablas = soup.find_all("table", class_="tabla-estadisticas")
        if len(tablas) >= 2:
            for idx, equipo_nombre in enumerate([equipo_local_nombre, equipo_visitante_nombre]):
                for fila in tablas[idx].find("tbody").find_all("tr", onclick=True):
                    onclick_attr = fila["onclick"]
                    match = re.search(r"(\{.*\})", onclick_attr)
                    if match:
                        json_str = match.group(1).replace("'", '"')
                        player_data = json.loads(json_str)
                        player_data['equipo'] = equipo_nombre
                        all_player_stats.append(player_data)
        if all_player_stats:
            df_box_scores = pd.DataFrame(all_player_stats)
            columnas = ['IdJugador', 'IdClub', 'IdEquipo', 'Nombre', 'NombreCompleto',
                        'Puntos', 'TirosDos', 'TirosTres', 'TirosLibres',
                        'ReboteDefensivo', 'ReboteOfensivo', 'RebotesTotales',
                        'Asistencias', 'Recuperaciones', 'Perdidas',
                        'TaponCometido','TaponRecibido', 'FaltaCometida','FaltaRecibida','Valoracion',
                        'TiempoJuego', 'CincoInicial', 'equipo']
            df_box_scores = df_box_scores[[col for col in columnas if col in df_box_scores.columns]]
            columnas_con_diccionarios = ['TirosDos', 'TirosTres', 'TirosLibres']
            for columna in columnas_con_diccionarios:
                if columna in df_box_scores.columns:
                    df_box_scores[f'{columna}Aciertos'] = df_box_scores[columna].apply(lambda x: x.get('Aciertos', 0))
                    df_box_scores[f'{columna}Fallos'] = df_box_scores[columna].apply(lambda x: x.get('Fallados', 0))
            df_box_scores = df_box_scores.drop(columns=[c for c in columnas_con_diccionarios if c in df_box_scores.columns])
        else:
            return None, None

        # --- PLAY BY PLAY ---
        driver.switch_to.default_content()
        driver.get(partido_link)
        wait = WebDriverWait(driver, 20)
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.TAG_NAME, "iframe")))
        en_vivo_tab = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.pestana-en-vivo")))
        driver.execute_script("arguments[0].click();", en_vivo_tab)
        iframe_pbp_selector = (By.CSS_SELECTOR, "div.contenido-en-vivo div:nth-child(2) iframe")
        wait.until(EC.frame_to_be_available_and_switch_to_it(iframe_pbp_selector))
        html_pbp = driver.page_source
        soup = BeautifulSoup(html_pbp, "html.parser")
        contenedor_acciones = soup.find("ul", class_="listadoAccionesPartido")
        if contenedor_acciones:
            acciones = contenedor_acciones.find_all("li", class_="accion")
            for accion in acciones:
                tipo_accion = jugador = cuarto = tiempo = ""
                puntos_local = None
                puntos_visita = None
                titulo_tag = accion.find("strong", class_="titulo")
                if titulo_tag:
                    tipo_accion = titulo_tag.get_text(strip=True)
                spans_info = accion.find_all("span", class_="informacion")
                if len(spans_info) >= 2:
                    jugador = spans_info[0].get_text(strip=True)
                    tiempo_text = spans_info[1].get_text(strip=True)
                    match = re.search(r"Cuarto\s*(\d+)\s*-\s*(\d{2}:\d{2}:\d{2})", tiempo_text)
                    if match:
                        cuarto = match.group(1)
                        tiempo = match.group(2)
                marcador_tag = accion.find("strong", class_="informacionAdicional")
                if marcador_tag:
                    marcador_texto = marcador_tag.get_text(strip=True)
                    partes_marcador = marcador_texto.split('-')
                    if len(partes_marcador) == 2:
                        try:
                            puntos_local = int(partes_marcador[0].strip())
                            puntos_visita = int(partes_marcador[1].strip())
                        except ValueError:
                            pass
                acciones_del_partido.append({
                    "cuarto": cuarto,
                    "tiempo": tiempo,
                    "accion": tipo_accion,
                    "jugador": jugador,
                    "puntos_local": puntos_local,
                    "puntos_visita": puntos_visita
                })
        if acciones_del_partido:
            df_acciones = pd.DataFrame(acciones_del_partido)
        else:
            return df_box_scores, None

        # --- GENERAR QUINTETOS ---
        df_box_scores["NombreCompleto_limpio"] = (
        df_box_scores["NombreCompleto"]
        .str.strip()          # elimina espacios adelante y atrás
        .str.replace(r"\s+", " ", regex=True)  # colapsa espacios dobles
          )

        df_box_scores["equipo_normalizado"] = (
        df_box_scores["equipo"]
        .str.replace(r"\s*\(.*\)?", "", regex=True)
        .str.strip()
          )

        player_to_team_map = pd.Series(df_box_scores.equipo_normalizado.values, index=df_box_scores.NombreCompleto_limpio).to_dict()
        nombre_local = df_box_scores['equipo_normalizado'].unique()[0]
        df_cronologico = df_acciones.iloc[::-1].reset_index(drop=True)
        jugadores_en_cancha = set()
        lista_de_quintetos_mixtos = []

        # === CAMBIO: variable para detectar inicio de cuarto
        cuarto_actual = None  # === CAMBIO

        for _, row in df_cronologico.iterrows():
            accion = row.get('accion', '')
            jugador = row.get('jugador', '')
            # === CAMBIO: leemos cuarto y tiempo para gatillar reset de formaciones al inicio de cada cuarto
            cuarto = row.get('cuarto', None)            # === CAMBIO
            tiempo_evento = row.get('tiempo', '') or '' # === CAMBIO

            # === CAMBIO: si cambia el cuarto y el tiempo es 00:10:00, reseteamos la cancha
            if pd.notna(cuarto) and cuarto != '' and cuarto != cuarto_actual:  # === CAMBIO
                jugadores_en_cancha = set()  # reset total de jugadores en cancha al inicio del cuarto  # === CAMBIO
                cuarto_actual = cuarto       # === CAMBIO

            accion_upper = accion.upper() if isinstance(accion, str) else ''
            if "ENTRA A PISTA" in accion_upper or "CAMBIO-ENTRA" in accion_upper:
                if pd.notna(jugador) and jugador != '':
                    jugadores_en_cancha.add(jugador)
            elif "ABANDONA LA PISTA" in accion_upper or "CAMBIO-SALE" in accion_upper:
                if pd.notna(jugador) and jugador != '':
                    jugadores_en_cancha.discard(jugador)
            lista_de_quintetos_mixtos.append(sorted(list(jugadores_en_cancha)))

        df_cronologico['quinteto_en_cancha'] = lista_de_quintetos_mixtos

        def dividir_quinteto(quinteto_mixto, roster_map, equipo_local_nombre):
            quinteto_local, quinteto_visita = [], []
            for jugador in quinteto_mixto:
                  jugador_limpio = (
                      str(jugador).strip()
                      .replace("  ", " ")
                  )
                  equipo = roster_map.get(jugador_limpio)
                  if equipo == equipo_local_nombre:
                      quinteto_local.append(jugador_limpio)
                  else:
                      quinteto_visita.append(jugador_limpio)
            return sorted(quinteto_local), sorted(quinteto_visita)

        nuevas_columnas = df_cronologico['quinteto_en_cancha'].apply(
            lambda q: pd.Series(dividir_quinteto(q, player_to_team_map, nombre_local))
        )
        nuevas_columnas.columns = ['quinteto_local', 'quinteto_visita']
        df_cronologico = pd.concat([df_cronologico, nuevas_columnas], axis=1)
        df_cronologico = df_cronologico.drop(columns=['quinteto_en_cancha'])
        df_acciones_final = df_cronologico.iloc[::-1].reset_index(drop=True)
        return df_box_scores, df_acciones_final
    except Exception as e:
        print(f"Error procesando partido: {e}")
        return None, None
    finally:
        if driver:
            driver.quit()

# --- PROCESAR LOS PRIMEROS 10 PARTIDOS ---
resultados_boxscore = [] # Lista original para Box Scores fusionados
resultados_pbp = []      # Nueva lista para DataFrames PBP

llaves_partidos = list(match_data.keys())[:10]

for partido_key in tqdm(llaves_partidos, desc="Procesando partidos"):
    partido_link = match_data[partido_key]["link_estadisticas"]
    df_box_scores, df_acciones_final = extraer_boxscore_y_pbp(partido_link)

    if df_box_scores is None or df_acciones_final is None:
        continue

    # *** NUEVO: Añadir 'partido_key' al DataFrame PBP ***
    # Esto es CRUCIAL para saber a qué partido pertenece cada acción
    df_acciones_final['partido_key'] = partido_key
    resultados_pbp.append(df_acciones_final.copy())

    # ----------------------------------------------------------------
    # El resto del código calcula y prepara el Box Score fusionado:
    # ----------------------------------------------------------------
    df_box_scores["NombreCompleto_limpio"] = (
    df_box_scores["NombreCompleto"]
    .str.strip()          # elimina espacios adelante y atrás
    .str.replace(r"\s+", " ", regex=True)  # colapsa espacios dobles
      )

    df_box_scores["equipo_normalizado"] = (
    df_box_scores["equipo"]
    .str.replace(r"\s*\(.*\)?", "", regex=True)
    .str.strip()
      )

    player_to_team_map = pd.Series(df_box_scores.equipo_normalizado.values, index=df_box_scores.NombreCompleto_limpio).to_dict()
    df_sorted = df_acciones_final.iloc[::-1].reset_index(drop=True)

    # ... (Cálculo de todas las estadísticas avanzadas como Plus/Minus, etc.) ...

    resultado_final = df_box_scores.copy()
    # ... (Proceso de merge y limpieza de Box Score/Stats Avanzadas) ...

    resultado_final['partido_key'] = partido_key
    resultados_boxscore.append(resultado_final) # Se guarda el Box Score fusionado

# --- CONCATENAR AMBOS RESULTADOS ---
if resultados_boxscore:
    # Concatenación de Box Scores (como ya estaba)
    df_boxscore_final = pd.concat(resultados_boxscore, ignore_index=True)
    print("\n✅ DataFrame final con todos los registros de Box Scores de los primeros 10 partidos.")
    # display(df_boxscore_final)

if resultados_pbp:
    # *** NUEVO: Concatenación del Play-by-Play ***
    df_pbp_final = pd.concat(resultados_pbp, ignore_index=True)
    print("✅ DataFrame final con todas las acciones Play-by-Play de los primeros 10 partidos.")
    # display(df_pbp_final) # Esta sería la base que buscas
else:
    print("No se pudieron procesar partidos correctamente.")

Procesando partidos: 100%|██████████| 10/10 [01:38<00:00,  9.85s/it]


✅ DataFrame final con todos los registros de Box Scores de los primeros 10 partidos.
✅ DataFrame final con todas las acciones Play-by-Play de los primeros 10 partidos.


prueba 10 partidos

In [ ]:
def extraer_boxscore_y_pbp(partido_link):
    """Extrae boxscore y play-by-play de un partido dado su link."""
    all_player_stats = []
    acciones_del_partido = []
    driver = None
    df_box_scores = None
    df_acciones_final = None

    try:
        # --- BOX SCORE ---
        options = Options()
        options.add_argument("--headless")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        driver.get(partido_link)
        wait = WebDriverWait(driver, 20)
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.TAG_NAME, "iframe")))
        tab_selector = (By.CSS_SELECTOR, "li.pestana-estadisticas")
        estadisticas_tab_element = wait.until(EC.visibility_of_element_located(tab_selector))
        driver.execute_script("arguments[0].click();", estadisticas_tab_element)
        time.sleep(1)
        iframe_de_datos_selector = (By.CSS_SELECTOR, "div.contenido-estadisticas.activo iframe")
        wait.until(EC.frame_to_be_available_and_switch_to_it(iframe_de_datos_selector))
        html_box_scores = driver.page_source
        soup = BeautifulSoup(html_box_scores, "html.parser")
        divs_nombres = soup.find_all("div", class_="nombre-equipo")
        equipo_local_nombre = divs_nombres[0].get_text(strip=True) if len(divs_nombres) >= 2 else "Local"
        equipo_visitante_nombre = divs_nombres[1].get_text(strip=True) if len(divs_nombres) >= 2 else "Visitante"
        tablas = soup.find_all("table", class_="tabla-estadisticas")
        if len(tablas) >= 2:
            for idx, equipo_nombre in enumerate([equipo_local_nombre, equipo_visitante_nombre]):
                for fila in tablas[idx].find("tbody").find_all("tr", onclick=True):
                    onclick_attr = fila["onclick"]
                    match = re.search(r"(\{.*\})", onclick_attr)
                    if match:
                        json_str = match.group(1).replace("'", '"')
                        player_data = json.loads(json_str)
                        player_data['equipo'] = equipo_nombre
                        all_player_stats.append(player_data)
        if all_player_stats:
            df_box_scores = pd.DataFrame(all_player_stats)
            columnas = ['IdJugador', 'IdClub', 'IdEquipo', 'Nombre', 'NombreCompleto',
                        'Puntos', 'TirosDos', 'TirosTres', 'TirosLibres',
                        'ReboteDefensivo', 'ReboteOfensivo', 'RebotesTotales',
                        'Asistencias', 'Recuperaciones', 'Perdidas',
                        'TaponCometido','TaponRecibido', 'FaltaCometida','FaltaRecibida','Valoracion',
                        'TiempoJuego', 'CincoInicial', 'equipo']
            df_box_scores = df_box_scores[[col for col in columnas if col in df_box_scores.columns]]
            columnas_con_diccionarios = ['TirosDos', 'TirosTres', 'TirosLibres']
            for columna in columnas_con_diccionarios:
                if columna in df_box_scores.columns:
                    df_box_scores[f'{columna}Aciertos'] = df_box_scores[columna].apply(lambda x: x.get('Aciertos', 0))
                    df_box_scores[f'{columna}Fallos'] = df_box_scores[columna].apply(lambda x: x.get('Fallados', 0))
            df_box_scores = df_box_scores.drop(columns=[c for c in columnas_con_diccionarios if c in df_box_scores.columns])
        else:
            return None, None

        # --- PLAY BY PLAY ---
        driver.switch_to.default_content()
        driver.get(partido_link)
        wait = WebDriverWait(driver, 20)
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.TAG_NAME, "iframe")))
        en_vivo_tab = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.pestana-en-vivo")))
        driver.execute_script("arguments[0].click();", en_vivo_tab)
        iframe_pbp_selector = (By.CSS_SELECTOR, "div.contenido-en-vivo div:nth-child(2) iframe")
        wait.until(EC.frame_to_be_available_and_switch_to_it(iframe_pbp_selector))
        html_pbp = driver.page_source
        soup = BeautifulSoup(html_pbp, "html.parser")
        contenedor_acciones = soup.find("ul", class_="listadoAccionesPartido")
        if contenedor_acciones:
            acciones = contenedor_acciones.find_all("li", class_="accion")
            for accion in acciones:
                tipo_accion = jugador = cuarto = tiempo = ""
                puntos_local = None
                puntos_visita = None
                titulo_tag = accion.find("strong", class_="titulo")
                if titulo_tag:
                    tipo_accion = titulo_tag.get_text(strip=True)
                spans_info = accion.find_all("span", class_="informacion")
                if len(spans_info) >= 2:
                    jugador = spans_info[0].get_text(strip=True)
                    tiempo_text = spans_info[1].get_text(strip=True)
                    match = re.search(r"Cuarto\s*(\d+)\s*-\s*(\d{2}:\d{2}:\d{2})", tiempo_text)
                    if match:
                        cuarto = match.group(1)
                        tiempo = match.group(2)
                marcador_tag = accion.find("strong", class_="informacionAdicional")
                if marcador_tag:
                    marcador_texto = marcador_tag.get_text(strip=True)
                    partes_marcador = marcador_texto.split('-')
                    if len(partes_marcador) == 2:
                        try:
                            puntos_local = int(partes_marcador[0].strip())
                            puntos_visita = int(partes_marcador[1].strip())
                        except ValueError:
                            pass
                acciones_del_partido.append({
                    "cuarto": cuarto,
                    "tiempo": tiempo,
                    "accion": tipo_accion,
                    "jugador": jugador,
                    "puntos_local": puntos_local,
                    "puntos_visita": puntos_visita
                })
        if acciones_del_partido:
            df_acciones = pd.DataFrame(acciones_del_partido)
        else:
            return df_box_scores, None

        df_box_scores["NombreCompleto_limpio"] = (
        df_box_scores["NombreCompleto"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        )

        df_box_scores["equipo_normalizado"] = (
            df_box_scores["equipo"]
            .str.replace(r"\s*\(.*\)?", "", regex=True)
            .str.strip()
        )

        player_to_team_map = pd.Series(
            df_box_scores["equipo_normalizado"].values,
            index=df_box_scores["NombreCompleto_limpio"]
        ).to_dict()

        # identificar equipos
        equipos = df_box_scores["equipo_normalizado"].unique()
        nombre_local = equipos[0]
        nombre_visitante = equipos[1] if len(equipos) > 1 else None

        # --- Orden cronológico inverso ---
        df_cronologico = df_acciones.iloc[::-1].reset_index(drop=True)

        jugadores_en_cancha = set()
        lista_quintetos = []
        cuarto_actual = None

        for _, row in df_cronologico.iterrows():
            accion = row.get("accion", "")
            jugador = row.get("jugador", "")
            cuarto = row.get("cuarto", None)

            accion_upper = accion.upper() if isinstance(accion, str) else ""

            # 🔁 Reset SOLO al inicio real de cada cuarto
            if pd.notna(cuarto) and cuarto != "" and cuarto != cuarto_actual:
                jugadores_en_cancha = set()
                cuarto_actual = cuarto

            # Cambios
            if "ENTRA A PISTA" in accion_upper or "CAMBIO-ENTRA" in accion_upper:
                if pd.notna(jugador) and jugador != "":
                    jugadores_en_cancha.add(jugador)

            elif "ABANDONA LA PISTA" in accion_upper or "CAMBIO-SALE" in accion_upper:
                if pd.notna(jugador) and jugador != "":
                    jugadores_en_cancha.discard(jugador)

            lista_quintetos.append(sorted(jugadores_en_cancha))

        df_cronologico["quinteto_en_cancha"] = lista_quintetos

        # --- Separar quintetos ---
        def dividir_quinteto(quinteto, roster_map, equipo_local):
            q_local, q_visita = [], []
            for j in quinteto:
                j_limpio = str(j).strip().replace("  ", " ")
                equipo = roster_map.get(j_limpio)
                if equipo == equipo_local:
                    q_local.append(j_limpio)
                else:
                    q_visita.append(j_limpio)
            return sorted(q_local), sorted(q_visita)

        nuevas_cols = df_cronologico["quinteto_en_cancha"].apply(
            lambda q: pd.Series(dividir_quinteto(q, player_to_team_map, nombre_local))
        )
        nuevas_cols.columns = ["quinteto_local", "quinteto_visita"]
        df_cronologico = pd.concat([df_cronologico, nuevas_cols], axis=1)

        # --- Cantidad de jugadores en cancha ---
        df_cronologico["jugadores_en_cancha"] = (
            df_cronologico["quinteto_local"].apply(len) +
            df_cronologico["quinteto_visita"].apply(len)
        )

        # =========================================================
        # === DEFINIR EQUIPO_ACCION DESDE EL CRUDO (CLAVE) ===
        # =========================================================

        df_cronologico["jugador_limpio"] = (
            df_cronologico["jugador"]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

        def identificar_equipo_accion(row):
            jugador = row.get("jugador_limpio")

            # 1️⃣ Por jugador (prioridad absoluta)
            if pd.notna(jugador) and jugador in player_to_team_map:
                return player_to_team_map[jugador]

            accion = str(row.get("accion", "")).upper()

            # 2️⃣ Por texto explícito
            if "LOCAL" in accion:
                return nombre_local
            if "VISITANTE" in accion:
                return nombre_visitante

            return None

        df_cronologico["equipo_accion"] = df_cronologico.apply(
            identificar_equipo_accion, axis=1
        )

        df_cronologico["tipo_equipo_accion"] = df_cronologico["equipo_accion"].apply(
            lambda x: "local" if x == nombre_local else (
                "visitante" if x == nombre_visitante else None
            )
        )

        # --- Limpieza final ---
        df_cronologico = df_cronologico.drop(columns=["quinteto_en_cancha"])

        # volver a orden original (última fila = última acción)
        df_acciones_final = df_cronologico.iloc[::-1].reset_index(drop=True)

        return df_box_scores, df_acciones_final

    except Exception as e:
        print(f"Error procesando partido: {e}")
        return None, None
    finally:
        if driver:
            driver.quit()

# --- PROCESAR LOS PRIMEROS 10 PARTIDOS ---
resultados_boxscore = [] # Lista original para Box Scores fusionados
resultados_pbp = []      # Nueva lista para DataFrames PBP

llaves_partidos = list(match_data.keys())[:10]

for partido_key in tqdm(llaves_partidos, desc="Procesando partidos"):
    partido_link = match_data[partido_key]["link_estadisticas"]
    df_box_scores, df_acciones_final = extraer_boxscore_y_pbp(partido_link)

    if df_box_scores is None or df_acciones_final is None:
        continue

    # *** NUEVO: Añadir 'partido_key' al DataFrame PBP ***
    # Esto es CRUCIAL para saber a qué partido pertenece cada acción
    df_acciones_final['partido_key'] = partido_key
    resultados_pbp.append(df_acciones_final.copy())

    # ----------------------------------------------------------------
    # El resto del código calcula y prepara el Box Score fusionado:
    # ----------------------------------------------------------------
    df_box_scores["NombreCompleto_limpio"] = (
    df_box_scores["NombreCompleto"]
    .str.strip()          # elimina espacios adelante y atrás
    .str.replace(r"\s+", " ", regex=True)  # colapsa espacios dobles
      )

    df_box_scores["equipo_normalizado"] = (
    df_box_scores["equipo"]
    .str.replace(r"\s*\(.*\)?", "", regex=True)
    .str.strip()
      )

    player_to_team_map = pd.Series(df_box_scores.equipo_normalizado.values, index=df_box_scores.NombreCompleto_limpio).to_dict()
    df_sorted = df_acciones_final.iloc[::-1].reset_index(drop=True)

    # ... (Cálculo de todas las estadísticas avanzadas como Plus/Minus, etc.) ...

    resultado_final = df_box_scores.copy()
    # ... (Proceso de merge y limpieza de Box Score/Stats Avanzadas) ...

    resultado_final['partido_key'] = partido_key
    resultados_boxscore.append(resultado_final) # Se guarda el Box Score fusionado

# --- CONCATENAR AMBOS RESULTADOS ---
if resultados_boxscore:
    # Concatenación de Box Scores (como ya estaba)
    df_boxscore_final = pd.concat(resultados_boxscore, ignore_index=True)
    print("\n✅ DataFrame final con todos los registros de Box Scores de los primeros 10 partidos.")
    # display(df_boxscore_final)

if resultados_pbp:
    # *** NUEVO: Concatenación del Play-by-Play ***
    df_pbp_final = pd.concat(resultados_pbp, ignore_index=True)
    print("✅ DataFrame final con todas las acciones Play-by-Play de los primeros 10 partidos.")
    # display(df_pbp_final) # Esta sería la base que buscas
else:
    print("No se pudieron procesar partidos correctamente.")

Procesando partidos: 100%|██████████| 10/10 [01:32<00:00,  9.25s/it]


✅ DataFrame final con todos los registros de Box Scores de los primeros 10 partidos.
✅ DataFrame final con todas las acciones Play-by-Play de los primeros 10 partidos.


In [ ]:
#df_pbp_final.to_excel("pbp_crudo_10_partidos.xlsx", index=False)

10 partidos - preprocesados

In [ ]:
# ---------------------------
# Procesar N partidos (primeros 10) -> PBP preprocesado concatenado
# ---------------------------

import unicodedata
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

# ---------- Utilidades (idénticas a las que venías usando) ----------
def quitar_tildes(texto):
    if isinstance(texto, str):
        return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

def normalizar_accion_simple(texto):
    s = str(texto).upper()
    s = quitar_tildes(s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def tiempo_a_segundos(t):
    try:
        hh, mm, ss = t.split(":")
        return int(hh) * 3600 + int(mm) * 60 + int(ss)
    except:
        return float('nan')

# ---------- Función de preprocesado por partido (basada en tu pipeline de 1 partido) ----------
def preprocesar_pbp_single(df_acciones_final, nombre_local, nombre_visitante):
    """Recibe df_acciones_final (raw del partido), nombre_local, nombre_visitante.
       Devuelve df_preprocesada listo para concatenar (misma estructura que tu chunk de 1 partido)."""
    df_preprocesada = df_acciones_final.copy()

    # Si no vienen los quintetos como listas en todas las filas, asumimos que tu función ya las genera correctamente.
    # A) crear jugadores_en_cancha si no existe (suma longitudes quintetos)
    def len_quinteto(x):
        try:
            return len(x) if x is not None else 0
        except Exception:
            return 0

    if 'jugadores_en_cancha' not in df_preprocesada.columns:
        df_preprocesada['jugadores_en_cancha'] = df_preprocesada.apply(
            lambda r: len_quinteto(r.get('quinteto_local')) + len_quinteto(r.get('quinteto_visita')), axis=1
        )

    # 1) Filtrar por jugadores en cancha >= 10
    if 'jugadores_en_cancha' in df_preprocesada.columns:
        df_preprocesada = df_preprocesada[df_preprocesada["jugadores_en_cancha"] >= 10].copy()

    # 2) Normalizar acción
    df_preprocesada['accion_norm'] = df_preprocesada['accion'].apply(normalizar_accion_simple)

    # 3) Eliminar acciones prohibidas
    strings_a_eliminar = [
        "ENTRA A PISTA",
        "INICIO DEL PERIODO",
        "FINAL DEL PERIODO",
        "ABANDONA LA PISTA",
        "TAPON REALIZADO",
        "TECNICA",
        "TECNICA BANQUILLO C",
        "TECNICA BANQUILLO B",
        "TECNICA DESCALIFICANTE",
        "TECNICA DESCALIFICANTE ACOMPAÑANTE NO PARTICIPA",
        "TECNICA-E (ALINEACIÓN ILEGAL)",
        "TIEMPO MUERTO SOLICITADO",
        "ANTIDEPORTIVA"
    ]
    escaped_strings = [re.escape(s) for s in strings_a_eliminar]
    patron = "|".join(escaped_strings)
    df_preprocesada = df_preprocesada[~df_preprocesada['accion_norm'].str.contains(patron, regex=True, na=False)].copy()

    # 4) FLECHA DE ALTERNANCIA -> voltear equipos si aparece
    mask_flecha = df_preprocesada['accion_norm'].str.contains('FLECHA ALTERNANCIA', na=False)
    if mask_flecha.any():
        df_preprocesada.loc[mask_flecha, "equipo_accion"] = df_preprocesada.loc[mask_flecha, "equipo_accion"].map({
            nombre_local: nombre_visitante,
            nombre_visitante: nombre_local
        })
        df_preprocesada.loc[mask_flecha, "tipo_equipo_accion"] = df_preprocesada.loc[mask_flecha, "tipo_equipo_accion"].map({
            "local": "visitante",
            "visitante": "local"
        })

    # 5) Eliminar últimos REBOTE DEFENSIVO del inicio de cuarto si corresponde
    mask_last_in_quarter = df_preprocesada['cuarto'].fillna(-999) != df_preprocesada['cuarto'].shift(1).fillna(-999)
    mask_rebote_def = df_preprocesada['accion_norm'].str.contains(r'\bREBOTE DEFENSIVO\b', regex=True, na=False)
    mask_remove_last_rebote = mask_last_in_quarter & mask_rebote_def
    if mask_remove_last_rebote.any():
        df_preprocesada = df_preprocesada.loc[~mask_remove_last_rebote].copy()

    # 6) Reglas para faltas (usa accion previa en orden cronológico inverso como antes)
    accion_prev = df_preprocesada['accion_norm'].shift(-1).fillna('')
    equipo_prev = df_preprocesada['equipo_accion'].shift(-1)

    mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)?\b', regex=True, na=False)
    mask_perdida_prev = accion_prev.str.contains(r'\bPERDIDA\b', regex=True, na=False)
    mask_asistencia_prev = accion_prev.str.contains(r'\bASISTENCIA\b', regex=True, na=False)
    mask_triple_prev = accion_prev.str.contains(r'\bTRIPLE\b', na=False)
    mask_canasta2_prev = accion_prev.str.contains(r'\bCANASTA\b', na=False) & accion_prev.str.contains(r'\b2\b', na=False)

    tiempo_prev_seg = df_preprocesada['tiempo'].shift(-1).apply(tiempo_a_segundos)
    tiempo_curr_seg = df_preprocesada['tiempo'].apply(tiempo_a_segundos)
    elapsed = tiempo_prev_seg - tiempo_curr_seg

    mask_scoring_prev_diff_time = (
        (mask_triple_prev | mask_canasta2_prev | mask_asistencia_prev)
        & (elapsed > 4)
    )

    mask_prev_excepcion = (
        mask_tiro_prev
        | mask_perdida_prev
        | mask_scoring_prev_diff_time
    )

    mask_falta = df_preprocesada['accion_norm'] == 'FALTA COMETIDA'
    mask_equipo_distinto = df_preprocesada['equipo_accion'] != equipo_prev
    mask_keep_falta = mask_falta & mask_prev_excepcion & mask_equipo_distinto
    mask_a_eliminar = mask_falta & (~mask_keep_falta)
    if mask_a_eliminar.any():
        df_preprocesada = df_preprocesada.loc[~mask_a_eliminar].copy()

    # limpieza final: drop helper
    df_preprocesada = df_preprocesada.drop(columns=['accion_norm'])

    # 7) Añadir posesion, quintetos_por_pos, pts_acum, pts_fin
    # invertir a orden cronológico real (fila 0 = primera acción)
    df_tmp = df_preprocesada.iloc[::-1].reset_index(drop=True)

    cond_nueva_posesion = (
        (df_tmp['equipo_accion'] != df_tmp['equipo_accion'].shift(1)) |
        (df_tmp['cuarto'] != df_tmp['cuarto'].shift(1))
    )
    cond_nueva_posesion.iloc[0] = True
    df_tmp['posesion'] = cond_nueva_posesion.cumsum().astype(int)

    def contar_quintetos(gr):
        pares = list(zip(gr['quinteto_local'].fillna('__NA__'),
                         gr['quinteto_visita'].fillna('__NA__')))
        return len(pd.Series(pares).drop_duplicates())

    quintetos_por_pos = df_tmp.groupby('posesion').apply(contar_quintetos)
    quintetos_por_pos.name = 'quintetos_por_pos'
    df_tmp = df_tmp.join(quintetos_por_pos, on='posesion')

    # pts acumulados por forward fill (empiezan en 0)
    df_tmp['pts_acum_local'] = df_tmp['puntos_local'].ffill().fillna(0).astype(int)
    df_tmp['pts_acum_visit'] = df_tmp['puntos_visita'].ffill().fillna(0).astype(int)

    # pts_fin en última fila de cada posesión (en orden cronológico real)
    es_ultima_fila_posesion = df_tmp['posesion'] != df_tmp['posesion'].shift(-1)
    df_tmp['pts_fin_loc'] = None
    df_tmp['pts_fin_vis'] = None
    df_tmp.loc[es_ultima_fila_posesion, 'pts_fin_loc'] = df_tmp.loc[es_ultima_fila_posesion, 'pts_acum_local']
    df_tmp.loc[es_ultima_fila_posesion, 'pts_fin_vis'] = df_tmp.loc[es_ultima_fila_posesion, 'pts_acum_visit']

    # volver al orden original (igual que tu df original: última fila = última acción)
    df_preprocesada = df_tmp.iloc[::-1].reset_index(drop=True)

    return df_preprocesada

# ---------------------------
# Loop principal: procesar primeros 10 partidos y concatenar PBP preprocesado
# ---------------------------

resultados_pbp_preprocesados = []
llaves_partidos = list(match_data.keys())[:10]

for partido_key in tqdm(llaves_partidos, desc="Procesando partidos"):
    partido_link = match_data[partido_key].get("link_estadisticas")
    if not partido_link:
        print(f"Sin link para partido {partido_key}, salteando.")
        continue

    # extraer (usa tu función existente)
    df_box_scores, df_acciones_final = extraer_boxscore_y_pbp(partido_link)

    if df_acciones_final is None:
        print(f"No se extrajo PBP para {partido_key}, salteando.")
        continue

    # inferir nombres de equipos si es posible (desde boxscore)
    nombre_local = None
    nombre_visitante = None
    if df_box_scores is not None and 'equipo' in df_box_scores.columns:
        equipos_unicos = list(df_box_scores['equipo'].unique())
        if len(equipos_unicos) >= 1:
            nombre_local = equipos_unicos[0]
            nombre_visitante = equipos_unicos[1] if len(equipos_unicos) > 1 else None

    # aplicar preprocesado del partido
    try:
        df_pre_proc = preprocesar_pbp_single(df_acciones_final, nombre_local, nombre_visitante)
    except Exception as e:
        print(f"Error preprocesando partido {partido_key}: {e}")
        continue

    # añadir identificador del partido y guardar
    df_pre_proc['partido_key'] = partido_key
    resultados_pbp_preprocesados.append(df_pre_proc.copy())

# concatenar todos los pbp preprocesados
if resultados_pbp_preprocesados:
    df_pbp_final_preproc = pd.concat(resultados_pbp_preprocesados, ignore_index=True)
    print("✅ PBP preprocesado concatenado para los partidos procesados.")
    display(df_pbp_final_preproc.head())
else:
    df_pbp_final_preproc = pd.DataFrame()
    print("⚠️ No se preprocesó ninguna PBP.")

# (opcional) guardar a disco
# df_pbp_final_preproc.to_excel("pbp_preprocesado_10_partidos.xlsx", index=False)


Procesando partidos:   0%|          | 0/10 [00:00<?, ?it/s]/tmp/ipython-input-1111174435.py:99: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)?\b', regex=True, na=False)
/tmp/ipython-input-1111174435.py:146: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  quintetos_por_pos = df_tmp.groupby('posesion').apply(contar_quintetos)
Procesando partidos:  10%|█         | 1/10 [00:10<01:31, 10.19s/it]/tmp/ipython-input-1111174435.py:99: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
 

✅ PBP preprocesado concatenado para los partidos procesados.


,cuarto,tiempo,accion,jugador,puntos_local,puntos_visita,quinteto_local,quinteto_visita,jugadores_en_cancha,jugador_limpio,equipo_accion,tipo_equipo_accion,posesion,quintetos_por_pos,pts_acum_local,pts_acum_visit,pts_fin_loc,pts_fin_vis,partido_key
0,4,00:00:21,PÉRDIDA DE BALÓN,"PROME, TIZIANO URIEL SEBASTIAN",NaN,NaN,"[ARAUJO, MAXIMO, LEMA, LEONARDO, MAIDANA, JERO...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"PROME, TIZIANO URIEL SEBASTIAN",BOCA,visitante,146,1,69,81,69,81,ATENAS (C) vs BOCA (007/10/2024 22:10)
1,4,00:00:32,REBOTE OFENSIVO#5,"GUERRERO MARGARIT, JUAN MARTIN",NaN,NaN,"[ARAUJO, MAXIMO, LEMA, LEONARDO, MAIDANA, JERO...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"GUERRERO MARGARIT, JUAN MARTIN",BOCA,visitante,146,1,69,81,None,None,ATENAS (C) vs BOCA (007/10/2024 22:10)
2,4,00:00:32,TIRO DE 3 FALLADO,"PROME, TIZIANO URIEL SEBASTIAN",NaN,NaN,"[ARAUJO, MAXIMO, LEMA, LEONARDO, MAIDANA, JERO...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"PROME, TIZIANO URIEL SEBASTIAN",BOCA,visitante,146,1,69,81,None,None,ATENAS (C) vs BOCA (007/10/2024 22:10)
3,4,00:00:55,TIRO LIBRE ANOTADO,"TOMATIS, TIAGO",69.0,81.0,"[ARAUJO, MAXIMO, LEMA, LEONARDO, MAIDANA, JERO...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"TOMATIS, TIAGO",ATENAS,local,145,1,69,81,69,81,ATENAS (C) vs BOCA (007/10/2024 22:10)
4,4,00:00:55,TIRO LIBRE ANOTADO,"TOMATIS, TIAGO",68.0,81.0,"[ARAUJO, MAXIMO, LEMA, LEONARDO, MAIDANA, JERO...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"TOMATIS, TIAGO",ATENAS,local,145,1,68,81,None,None,ATENAS (C) vs BOCA (007/10/2024 22:10)


In [ ]:
#df_pbp_final_preproc.to_excel("pbp_preprocesado_10_partidos.xlsx", index=False)

10 partidos - poss by poss

In [ ]:
# ---------------------------
# CREAR poss-by-poss concatenado para todos los partidos en df_pbp_final_preproc
# ---------------------------

import ast
import numpy as np
import pandas as pd

# ---- 0) chequeo rápido ----
if 'df_pbp_final_preproc' not in globals():
    raise RuntimeError("No encuentro 'df_pbp_final_preproc' en el entorno. Crea/concatena primero los PBP preprocesados.")

df_all = df_pbp_final_preproc.copy()

# ---- 1) parser robusto para quintetos (lista real, np.ndarray, string "['A','B']", None, etc.) ----
def parse_quinteto_python_list(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return [str(i).strip() for i in list(x) if pd.notna(i) and str(i).strip()!='']
    s = str(x).strip()
    if s.lower() in ('nan', 'none', ''):
        return []
    if s.startswith('[') and s.endswith(']'):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set, np.ndarray)):
                return [str(i).strip() for i in list(parsed) if pd.notna(i) and str(i).strip()!='']
        except Exception:
            pass
    return [s]

# ---- 2) construir lista UNIÓN de jugadores en todos los partidos (jugador + quintetos) ----
jugadores_from_col = df_all['jugador'].dropna().astype(str).unique().tolist()

todos_q = []
for q in pd.concat([df_all['quinteto_local'].fillna('[]'), df_all['quinteto_visita'].fillna('[]')], ignore_index=True):
    todos_q.extend(parse_quinteto_python_list(q))

jugadores_union = sorted(set(jugadores_from_col + todos_q))

# ---- 3) función para procesar un único partido (devuelve poss-by-poss de ese partido) ----
def procesar_partido_poss(df_partido):
    # asumimos que df_partido ya tiene columnas: 'posesion','pts_fin_loc','pts_fin_vis','quinteto_local','quinteto_visita',
    # 'equipo_accion','tipo_equipo_accion','equipo_local','equipo_visitante','jugador', etc.
    dfp = df_partido.copy().reset_index(drop=True)

    # quedarnos sólo con filas finales de posesión
    df_poss_rows = dfp[dfp['pts_fin_loc'].notna() | dfp['pts_fin_vis'].notna()].copy().reset_index(drop=True)
    if df_poss_rows.empty:
        return pd.DataFrame()  # no hay posesiones en este partido

    # parsear quintetos
    df_poss_rows['q_local_list'] = df_poss_rows['quinteto_local'].apply(parse_quinteto_python_list)
    df_poss_rows['q_vis_list']   = df_poss_rows['quinteto_visita'].apply(parse_quinteto_python_list)

    # asegurar numericidad
    for c in ['pts_fin_loc','pts_fin_vis','pts_acum_local','pts_acum_visit']:
        if c in df_poss_rows.columns:
            df_poss_rows[c] = pd.to_numeric(df_poss_rows[c], errors='coerce')

    # ordenar por posesion asc
    if 'posesion' in df_poss_rows.columns:
        df_poss_rows = df_poss_rows.sort_values('posesion').reset_index(drop=True)
    else:
        # si no hay posesion, crear una incremental (seguro que tu preprocesado la generó, pero por si acaso)
        df_poss_rows = df_poss_rows.reset_index(drop=True)
        df_poss_rows['posesion'] = np.arange(1, len(df_poss_rows)+1)

    # determinar si ataca local (preferir tipo_equipo_accion, sino comparar nombres de equipo)
    def ataca_local_row(r):
        tipo = r.get('tipo_equipo_accion')
        equipo_acc = r.get('equipo_accion')
        equipo_local = r.get('equipo_local')
        if isinstance(tipo, str) and tipo.strip() != '':
            t = tipo.lower()
            if 'local' in t:
                return True
            if 'visit' in t:
                return False
        return (pd.notna(equipo_acc) and pd.notna(equipo_local) and str(equipo_acc) == str(equipo_local))

    df_poss_rows['ataca_local'] = df_poss_rows.apply(ataca_local_row, axis=1)

    # preparar pts_fin rellenados y previous
    df_poss_rows['pts_fin_loc_f'] = df_poss_rows['pts_fin_loc'].fillna(0).astype(int)
    df_poss_rows['pts_fin_vis_f'] = df_poss_rows['pts_fin_vis'].fillna(0).astype(int)

    prev_pts_fin_loc = df_poss_rows['pts_fin_loc_f'].shift(1).fillna(0).astype(int)
    prev_pts_fin_vis = df_poss_rows['pts_fin_vis_f'].shift(1).fillna(0).astype(int)

    # calcular puntos por posesión según regla (diferencia con posesion anterior)
    df_poss_rows['puntos_pos'] = np.where(
        df_poss_rows['ataca_local'],
        (df_poss_rows['pts_fin_loc_f'] - prev_pts_fin_loc).astype(int),
        (df_poss_rows['pts_fin_vis_f'] - prev_pts_fin_vis).astype(int)
    ).astype(int)

    # determinar atacantes/defensores por fila (usar equipo_accion comparado con equipo_local/visitante preferentemente)
    def obtener_atac_def(row):
        q_local = set(row['q_local_list'])
        q_vis = set(row['q_vis_list'])
        equipo_acc = row.get('equipo_accion')
        equipo_local = row.get('equipo_local')
        equipo_visit = row.get('equipo_visitante')
        tipo_equipo_acc = row.get('tipo_equipo_accion', None)

        if pd.notna(equipo_acc) and pd.notna(equipo_local) and str(equipo_acc) == str(equipo_local):
            return q_local, q_vis
        if pd.notna(equipo_acc) and pd.notna(equipo_visit) and str(equipo_acc) == str(equipo_visit):
            return q_vis, q_local

        if isinstance(tipo_equipo_acc, str):
            t = tipo_equipo_acc.lower()
            if 'local' in t:
                return q_local, q_vis
            if 'visit' in t:
                return q_vis, q_local

        return q_local, q_vis

    # construir indicadores (usar jugadores_union para columnas estables)
    indic_rows = []
    for _, row in df_poss_rows.iterrows():
        atac, defen = obtener_atac_def(row)
        # inicializo en 0 para toda la unión de jugadores
        indic = {p: 0 for p in jugadores_union}
        for p in atac:
            if p in indic:
                indic[p] = 1
        for p in defen:
            if p in indic:
                indic[p] = -1
        indic_rows.append(indic)

    df_ind = pd.DataFrame(indic_rows, columns=jugadores_union)

    # concatenar: conservar todas las columnas originales + puntos_pos + indicadores
    cols_originales = [c for c in df_poss_rows.columns if c not in ('q_local_list','q_vis_list','pts_fin_loc_f','pts_fin_vis_f')]
    # asegurar columnas claves
    if 'puntos_pos' not in cols_originales:
        cols_originales = cols_originales + ['puntos_pos']
    if 'ataca_local' not in cols_originales:
        cols_originales = cols_originales + ['ataca_local']

    df_out = pd.concat([df_poss_rows[cols_originales].reset_index(drop=True), df_ind.reset_index(drop=True)], axis=1)

    return df_out

# ---- 4) procesar partido por partido y concatenar ----
dfs_poss = []
for partido_key, df_group in df_all.groupby('partido_key'):
    df_result = procesar_partido_poss(df_group)
    if df_result.empty:
        print(f"Advertencia: partido {partido_key} no generó posesiones.")
        continue
    # mantener partido_key (si se perdió)
    df_result['partido_key'] = partido_key
    dfs_poss.append(df_result)

if dfs_poss:
    df_poss_by_poss_all = pd.concat(dfs_poss, ignore_index=True, sort=False)
else:
    df_poss_by_poss_all = pd.DataFrame()

# ---- 5) limpieza final: ordenar columnas (info primero, luego jugadores) ----
if not df_poss_by_poss_all.empty:
    info_cols = [c for c in df_poss_by_poss_all.columns if c not in jugadores_union]
    df_poss_by_poss_all = df_poss_by_poss_all[info_cols + jugadores_union]

# ---- 6) mostrar / guardar ----
print("Poss-by-poss total filas:", len(df_poss_by_poss_all))
display(df_poss_by_poss_all.head(20))

# opcional: guardar
df_poss_by_poss_all.to_excel("poss_by_poss_10_partidos.xlsx", index=False)
# df_poss_by_poss_all.to_csv("poss_by_poss_10_partidos.csv", index=False)


Poss-by-poss total filas: 1595


,cuarto,tiempo,accion,jugador,puntos_local,puntos_visita,quinteto_local,quinteto_visita,jugadores_en_cancha,jugador_limpio,equipo_accion,tipo_equipo_accion,posesion,quintetos_por_pos,pts_acum_local,pts_acum_visit,pts_fin_loc,pts_fin_vis,partido_key,ataca_local,puntos_pos,,"AALIYA, LEE ABRAHAM","ACUÑA, ROBERTO SANTIAGO","AGUIRRE, GUSTAVO NICOLAS","ALESSIO, JOSE IGNACIO JESUS","ALIENDE, GUILLERMO ALFONZO","ALORDA, FRANCO","ANDERSON, ALPHONSO JORDAN","ANDUJAR, LUCAS","APREA, JULIAN PABLO","ARAUJO, MAXIMO","ARN BUSTAMANTE, LUCAS MARTIN","ARN BUSTAMANTE, LUCAS MARTIN","BANYARD, NICHOLAS RYAN","BARRAL, PEDRO","BARRALES, SANTIAGO","BARRIONUEVO, FELIPE","BARROS, AGUSTIN","BASUALDO, IVAN NICOLAS","BEDNAREK, IGNACIO NICOLAS","BELL, RANDY TYREE","BOMBINO PARADA, PEDRO ANTONIO","BOND JR, TIMOTHY LAMONT","BORSATTI, MATIAS DANIEL","BUEMO, CARLOS EMANUEL","BUENDIA, CARLOS MANUEL","CABRERA, MARTIN ANDRES","CAFFARO, ESTEBAN","CALFANI PERSINCULA, MATHIAS KENY","CAPPONI, FEBO LORENZO","CARTER, MYLES JUSTIN","CECI, MATEO","CHACON TIRADO, MARCOS","CHAMORRO, FACUNDO","CHAPERO, TOMAS","CHIARAVIGLIO, NICOLAS","CIFUENTES ROTTA, FIDEL","CISNEROS SIERRA, ANYELO JOSE","CLARKE, CHRISTOPHER ASHTON","COPELLO, NICOLAS","CORDOBA, GASTON EZEQUIEL","COSOLITO, MAURO NICOLAS","CUELLO, MARTIN NICOLAS","DATO, MARTINIANO","DE LOS SANTOS, NICOLAS","DEFELIPPO, JOSE","DELIA, MARCOS NICOLAS","DIAZ GIMENEZ, JORGE LUCIANO","DIEZ, ALEJANDRO GABRIEL","DIGGS, AVERY GERELL","DIGON, ALEJO ADRIEL","ELIAS, GASTON FEDERICO","FACELLO, AGUSTIN PEDRO","FERNANDEZ, VICTOR LUIS","FERRI, VALENTINO","FIELDS, CALEB JOSEPH","FIERRO, MARIANO","FILIPPETTI SATURNO, ENZO MATEO","FLOR, ERIC FABIAN","FRAGA PEREZ, LAUTARO EZEQUIEL","FRANCA DOS SANTOS, JOAO VITOR","FRANCHELA, TOBIAS","GALLEGOS, RODRIGO AGUSTIN","GARCIA GUERRERO, JORGE LUIS","GILETTO, SALVADOR","GOICOECHEA, MARIANO EZEQUIEL","GOMEZ QUINTEROS, CAMILO TAIEL","GRAMAJO, PABLO IVAN","GRUN, FEDERICO JOSE","GUERRA CAÑATE, YEFERSON ANTONIO","GUERRERO MARGARIT, JUAN MARTIN","GUERRERO, JERONIMO","HOOPER, CHRISTOPHER JALEEL","HURE, DANIEL HERNAN","IANGUAS, PEDRO HENRIQUE","IBARGUEN ANDREWS, ANDRES FELIPE","INYACO TOLEDO, FELIPE ANTONIO","JAIME, ANDRES JOAQUIN","JARA, CESAR AGUSTIN","JENKINS, JALEN KEMAL","KONSZTADT, ALEJANDRO","LADO, MOGGA CHARLES ARNADO","LARRAZA, JUAN IGNACIO","LATORRE, LUCAS","LEMA, LEONARDO","LEZCANO, EMILIANO","LOCKETT, PHILLIP DOMINIQUE DANIEL","LOPEZ LORENZ, JOAQUIN BAUTISTA","LUGARINI, BAUTISTA","LUGO, SEBASTIAN EZEQUIEL","MAGGI, ALEJO","MAIDANA, JERONIMO","MARCUCCI, NICOLAS JAVIER","MARTINEZ, JANO DAVID","MAZZUCCHELLI, SANTINO","MERCHANT, EDGAR HENRY","MONACCHI, TOMAS","MONTERO, JOSE IGNACIO","MORALES PEREYRA, AUGUSTO","MORALES, JUAN SEBASTIAN","MORALES, JULIAN","MOUSSA, FAUSTO","NALLY, JUAN TOMAS","NATION, ANDRE JAMES","NEGRETE, ALEX","NOBLEGA, JOAQUIN EXEQUIEL","OBENG MENSAH, YAW","OBERTO, JUAN CRUZ MATEO","PAUTASSO, AGUSTIN","PAZ, EZEQUIEL","PEREZ BAZAN, LUCAS NAIM","PEREZ DOUTHAT, SANTIAGO","PIÑERO, FACUNDO","PIÑERO, PATRICIO","POMOLI LARRABURU, NICOLA","PROME, TIZIANO URIEL SEBASTIAN","REESE V, JAMES LANARD","RIVERO, CARLOS EZEQUIEL","RODRIGUEZ, CAMILO","RODRIGUEZ, FELIPE SALVADOR","ROMEGIALLI, SANTINO ALBERTO","ROVERES, AUGUSTO DANIEL","RUTENBERG, FACUNDO OSVALDO","SAFAR, SELEM","SAIZ, JAVIER ALEJANDRO","SCHATTMANN, CARLOS LEONEL","SHRIVER, DAVID RANDALL","SMANIOTTI, FRANCO NAHUEL","SPANO, TOMAS","STENTA, NICOLAS","THOMAS JR, MARCUS WILEY","THORNTON, WILLIE ALFORD","THYGESEN, FERMIN VALDEMAR","TOLOSA, FACUNDO GERMAN","TOMATIS, TIAGO","TORETTA, EMILIANO","VALDIVIESO, AYRTON","VALLEJOS, ROLANDO EXEQUIEL","VEGA, SEBASTIAN ERNESTO","VENEGAS SCHAEFER, JUAN PABLO","VILDOZA, JOSE IGNACIO","VILDOZA, LEANDRO","WALLACE, DEVANTE RASHAD-KEITH","WALTON, ZACHERY CHRISTOPHER"
0,1,00:09:43,TIRO DE 2 FALLADO,"OBERTO, JUAN CRUZ MATEO",NaN,NaN,"[ARN BUSTAMANTE, LUCAS MARTIN, BUEMO, CARLOS E...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"OBERTO, JUAN CRUZ MATEO",ATENAS,local,1,1,0,0,0,0,ATENAS (C) vs BOCA (007/10/2024

Toda la temporada - cruda

In [ ]:
def extraer_boxscore_y_pbp(partido_link):
    """Extrae boxscore y play-by-play de un partido dado su link."""
    all_player_stats = []
    acciones_del_partido = []
    driver = None
    df_box_scores = None
    df_acciones_final = None

    try:
        # --- BOX SCORE ---
        options = Options()
        options.add_argument("--headless")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        driver.get(partido_link)
        wait = WebDriverWait(driver, 20)
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.TAG_NAME, "iframe")))
        tab_selector = (By.CSS_SELECTOR, "li.pestana-estadisticas")
        estadisticas_tab_element = wait.until(EC.visibility_of_element_located(tab_selector))
        driver.execute_script("arguments[0].click();", estadisticas_tab_element)
        time.sleep(1)
        iframe_de_datos_selector = (By.CSS_SELECTOR, "div.contenido-estadisticas.activo iframe")
        wait.until(EC.frame_to_be_available_and_switch_to_it(iframe_de_datos_selector))
        html_box_scores = driver.page_source
        soup = BeautifulSoup(html_box_scores, "html.parser")
        divs_nombres = soup.find_all("div", class_="nombre-equipo")
        equipo_local_nombre = divs_nombres[0].get_text(strip=True) if len(divs_nombres) >= 2 else "Local"
        equipo_visitante_nombre = divs_nombres[1].get_text(strip=True) if len(divs_nombres) >= 2 else "Visitante"
        tablas = soup.find_all("table", class_="tabla-estadisticas")
        if len(tablas) >= 2:
            for idx, equipo_nombre in enumerate([equipo_local_nombre, equipo_visitante_nombre]):
                for fila in tablas[idx].find("tbody").find_all("tr", onclick=True):
                    onclick_attr = fila["onclick"]
                    match = re.search(r"(\{.*\})", onclick_attr)
                    if match:
                        json_str = match.group(1).replace("'", '"')
                        player_data = json.loads(json_str)
                        player_data['equipo'] = equipo_nombre
                        all_player_stats.append(player_data)
        if all_player_stats:
            df_box_scores = pd.DataFrame(all_player_stats)
            columnas = ['IdJugador', 'IdClub', 'IdEquipo', 'Nombre', 'NombreCompleto',
                        'Puntos', 'TirosDos', 'TirosTres', 'TirosLibres',
                        'ReboteDefensivo', 'ReboteOfensivo', 'RebotesTotales',
                        'Asistencias', 'Recuperaciones', 'Perdidas',
                        'TaponCometido','TaponRecibido', 'FaltaCometida','FaltaRecibida','Valoracion',
                        'TiempoJuego', 'CincoInicial', 'equipo']
            df_box_scores = df_box_scores[[col for col in columnas if col in df_box_scores.columns]]
            columnas_con_diccionarios = ['TirosDos', 'TirosTres', 'TirosLibres']
            for columna in columnas_con_diccionarios:
                if columna in df_box_scores.columns:
                    df_box_scores[f'{columna}Aciertos'] = df_box_scores[columna].apply(lambda x: x.get('Aciertos', 0))
                    df_box_scores[f'{columna}Fallos'] = df_box_scores[columna].apply(lambda x: x.get('Fallados', 0))
            df_box_scores = df_box_scores.drop(columns=[c for c in columnas_con_diccionarios if c in df_box_scores.columns])
        else:
            return None, None

        # --- PLAY BY PLAY ---
        driver.switch_to.default_content()
        driver.get(partido_link)
        wait = WebDriverWait(driver, 20)
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.TAG_NAME, "iframe")))
        en_vivo_tab = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.pestana-en-vivo")))
        driver.execute_script("arguments[0].click();", en_vivo_tab)
        iframe_pbp_selector = (By.CSS_SELECTOR, "div.contenido-en-vivo div:nth-child(2) iframe")
        wait.until(EC.frame_to_be_available_and_switch_to_it(iframe_pbp_selector))
        html_pbp = driver.page_source
        soup = BeautifulSoup(html_pbp, "html.parser")
        contenedor_acciones = soup.find("ul", class_="listadoAccionesPartido")
        if contenedor_acciones:
            acciones = contenedor_acciones.find_all("li", class_="accion")
            for accion in acciones:
                tipo_accion = jugador = cuarto = tiempo = ""
                puntos_local = None
                puntos_visita = None
                titulo_tag = accion.find("strong", class_="titulo")
                if titulo_tag:
                    tipo_accion = titulo_tag.get_text(strip=True)
                spans_info = accion.find_all("span", class_="informacion")
                if len(spans_info) >= 2:
                    jugador = spans_info[0].get_text(strip=True)
                    tiempo_text = spans_info[1].get_text(strip=True)
                    match = re.search(r"Cuarto\s*(\d+)\s*-\s*(\d{2}:\d{2}:\d{2})", tiempo_text)
                    if match:
                        cuarto = match.group(1)
                        tiempo = match.group(2)
                marcador_tag = accion.find("strong", class_="informacionAdicional")
                if marcador_tag:
                    marcador_texto = marcador_tag.get_text(strip=True)
                    partes_marcador = marcador_texto.split('-')
                    if len(partes_marcador) == 2:
                        try:
                            puntos_local = int(partes_marcador[0].strip())
                            puntos_visita = int(partes_marcador[1].strip())
                        except ValueError:
                            pass
                acciones_del_partido.append({
                    "cuarto": cuarto,
                    "tiempo": tiempo,
                    "accion": tipo_accion,
                    "jugador": jugador,
                    "puntos_local": puntos_local,
                    "puntos_visita": puntos_visita
                })
        if acciones_del_partido:
            df_acciones = pd.DataFrame(acciones_del_partido)
        else:
            return df_box_scores, None

        df_box_scores["NombreCompleto_limpio"] = (
        df_box_scores["NombreCompleto"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        )

        df_box_scores["equipo_normalizado"] = (
            df_box_scores["equipo"]
            .str.replace(r"\s*\(.*\)?", "", regex=True)
            .str.strip()
        )

        player_to_team_map = pd.Series(
            df_box_scores["equipo_normalizado"].values,
            index=df_box_scores["NombreCompleto_limpio"]
        ).to_dict()

        # identificar equipos
        equipos = df_box_scores["equipo_normalizado"].unique()
        nombre_local = equipos[0]
        nombre_visitante = equipos[1] if len(equipos) > 1 else None

        # --- Orden cronológico inverso ---
        df_cronologico = df_acciones.iloc[::-1].reset_index(drop=True)

        jugadores_en_cancha = set()
        lista_quintetos = []
        cuarto_actual = None

        for _, row in df_cronologico.iterrows():
            accion = row.get("accion", "")
            jugador = row.get("jugador", "")
            cuarto = row.get("cuarto", None)

            accion_upper = accion.upper() if isinstance(accion, str) else ""

            # 🔁 Reset SOLO al inicio real de cada cuarto
            if pd.notna(cuarto) and cuarto != "" and cuarto != cuarto_actual:
                jugadores_en_cancha = set()
                cuarto_actual = cuarto

            # Cambios
            if "ENTRA A PISTA" in accion_upper or "CAMBIO-ENTRA" in accion_upper:
                if pd.notna(jugador) and jugador != "":
                    jugadores_en_cancha.add(jugador)

            elif "ABANDONA LA PISTA" in accion_upper or "CAMBIO-SALE" in accion_upper:
                if pd.notna(jugador) and jugador != "":
                    jugadores_en_cancha.discard(jugador)

            lista_quintetos.append(sorted(jugadores_en_cancha))

        df_cronologico["quinteto_en_cancha"] = lista_quintetos

                # --- Separar quintetos ---
        def dividir_quinteto(quinteto, roster_map, equipo_local):
            q_local, q_visita = [], []
            for j in quinteto:
                j_limpio = str(j).strip().replace("  ", " ")
                equipo = roster_map.get(j_limpio)
                if equipo == equipo_local:
                    q_local.append(j_limpio)
                else:
                    q_visita.append(j_limpio)
            return sorted(q_local), sorted(q_visita)

        nuevas_cols = df_cronologico["quinteto_en_cancha"].apply(
            lambda q: pd.Series(dividir_quinteto(q, player_to_team_map, nombre_local))
        )
        nuevas_cols.columns = ["quinteto_local", "quinteto_visita"]
        df_cronologico = pd.concat([df_cronologico, nuevas_cols], axis=1)

        # --- Cantidad de jugadores en cancha ---
        df_cronologico["jugadores_en_cancha"] = (
            df_cronologico["quinteto_local"].apply(len) +
            df_cronologico["quinteto_visita"].apply(len)
        )

        # =========================================================
        # === DEFINIR EQUIPO_ACCION DESDE EL CRUDO (CLAVE) ===
        # =========================================================

        df_cronologico["jugador_limpio"] = (
            df_cronologico["jugador"]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

        def identificar_equipo_accion(row):
            jugador = row.get("jugador_limpio")

            # 1️⃣ Por jugador (prioridad absoluta)
            if pd.notna(jugador) and jugador in player_to_team_map:
                return player_to_team_map[jugador]

            accion = str(row.get("accion", "")).upper()

            # 2️⃣ Por texto explícito
            if "LOCAL" in accion:
                return nombre_local
            if "VISITANTE" in accion:
                return nombre_visitante

            return None

        df_cronologico["equipo_accion"] = df_cronologico.apply(
            identificar_equipo_accion, axis=1
        )

        df_cronologico["tipo_equipo_accion"] = df_cronologico["equipo_accion"].apply(
            lambda x: "local" if x == nombre_local else (
                "visitante" if x == nombre_visitante else None
            )
        )

        # --- Limpieza final ---
        df_cronologico = df_cronologico.drop(columns=["quinteto_en_cancha"])

        # volver a orden original (última fila = última acción)
        df_acciones_final = df_cronologico.iloc[::-1].reset_index(drop=True)

        return df_box_scores, df_acciones_final

    except Exception as e:
        print(f"Error procesando partido: {e}")
        return None, None
    finally:
        if driver:
            driver.quit()

# --- PROCESAR TODOS LOS PARTIDOS ---
resultados_boxscore = [] # Lista original para Box Scores fusionados
resultados_pbp = []      # Nueva lista para DataFrames PBP

llaves_partidos = list(match_data.keys())

for partido_key in tqdm(llaves_partidos, desc="Procesando partidos"):
    partido_link = match_data[partido_key]["link_estadisticas"]
    df_box_scores, df_acciones_final = extraer_boxscore_y_pbp(partido_link)

    if df_box_scores is None or df_acciones_final is None:
        continue

    # *** NUEVO: Añadir 'partido_key' al DataFrame PBP ***
    # Esto es CRUCIAL para saber a qué partido pertenece cada acción
    df_acciones_final['partido_key'] = partido_key
    resultados_pbp.append(df_acciones_final.copy())

    resultado_final = df_box_scores.copy()
    # ... (Proceso de merge y limpieza de Box Score/Stats Avanzadas) ...

    resultado_final['partido_key'] = partido_key
    resultados_boxscore.append(resultado_final) # Se guarda el Box Score fusionado

# --- CONCATENAR AMBOS RESULTADOS ---
if resultados_boxscore:
    # Concatenación de Box Scores (como ya estaba)
    df_boxscore_final = pd.concat(resultados_boxscore, ignore_index=True)
    print("\n✅ DataFrame final con todos los registros de Box Scores.")
    # display(df_boxscore_final)

if resultados_pbp:
    # *** NUEVO: Concatenación del Play-by-Play ***
    df_pbp_final = pd.concat(resultados_pbp, ignore_index=True)
    print("✅ DataFrame final con todas las acciones Play-by-Plays.")
    # display(df_pbp_final) # Esta sería la base que buscas
else:
    print("No se pudieron procesar partidos correctamente.")

Procesando partidos: 100%|██████████| 380/380 [1:40:42<00:00, 15.90s/it]



✅ DataFrame final con todos los registros de Box Scores.
✅ DataFrame final con todas las acciones Play-by-Plays.


In [ ]:
df_boxscore_final.to_csv("df_boxscore_final.csv", index=False)

Box score - toda la temporada

In [ ]:
def extraer_boxscore(partido_link):
    """Extrae boxscore y play-by-play de un partido dado su link."""
    all_player_stats = []
    driver = None
    df_box_scores = None

    try:
        # --- BOX SCORE ---
        options = Options()
        options.add_argument("--headless")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        driver.get(partido_link)
        wait = WebDriverWait(driver, 20)
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.TAG_NAME, "iframe")))
        tab_selector = (By.CSS_SELECTOR, "li.pestana-estadisticas")
        estadisticas_tab_element = wait.until(EC.visibility_of_element_located(tab_selector))
        driver.execute_script("arguments[0].click();", estadisticas_tab_element)
        time.sleep(1)
        iframe_de_datos_selector = (By.CSS_SELECTOR, "div.contenido-estadisticas.activo iframe")
        wait.until(EC.frame_to_be_available_and_switch_to_it(iframe_de_datos_selector))
        html_box_scores = driver.page_source
        soup = BeautifulSoup(html_box_scores, "html.parser")
        divs_nombres = soup.find_all("div", class_="nombre-equipo")
        equipo_local_nombre = divs_nombres[0].get_text(strip=True) if len(divs_nombres) >= 2 else "Local"
        equipo_visitante_nombre = divs_nombres[1].get_text(strip=True) if len(divs_nombres) >= 2 else "Visitante"
        tablas = soup.find_all("table", class_="tabla-estadisticas")
        if len(tablas) >= 2:
            for idx, equipo_nombre in enumerate([equipo_local_nombre, equipo_visitante_nombre]):
                for fila in tablas[idx].find("tbody").find_all("tr", onclick=True):
                    onclick_attr = fila["onclick"]
                    match = re.search(r"({\S*})", onclick_attr)
                    if match:
                        json_str = match.group(1).replace("'", '"')
                        player_data = json.loads(json_str)
                        player_data['equipo'] = equipo_nombre
                        all_player_stats.append(player_data)
        if all_player_stats:
            df_box_scores = pd.DataFrame(all_player_stats)
            columnas = ['IdJugador', 'IdClub', 'IdEquipo', 'Nombre', 'NombreCompleto',
                        'Puntos', 'TirosDos', 'TirosTres', 'TirosLibres',
                        'ReboteDefensivo', 'ReboteOfensivo', 'RebotesTotales',
                        'Asistencias', 'Recuperaciones', 'Perdidas',
                        'TaponCometido','TaponRecibido', 'FaltaCometida','FaltaRecibida','Valoracion',
                        'TiempoJuego', 'CincoInicial', 'equipo']
            df_box_scores = df_box_scores[[col for col in columnas if col in df_box_scores.columns]]
            columnas_con_diccionarios = ['TirosDos', 'TirosTres', 'TirosLibres']
            for columna in columnas_con_diccionarios:
                if columna in df_box_scores.columns:
                    df_box_scores[f'{columna}Aciertos'] = df_box_scores[columna].apply(lambda x: x.get('Aciertos', 0))
                    df_box_scores[f'{columna}Fallos'] = df_box_scores[columna].apply(lambda x: x.get('Fallados', 0))
            df_box_scores = df_box_scores.drop(columns=[c for c in columnas_con_diccionarios if c in df_box_scores.columns])

            # Move these lines inside the 'if all_player_stats' block
            df_box_scores["NombreCompleto_limpio"] = (
            df_box_scores["NombreCompleto"]
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            )

            df_box_scores["equipo_normalizado"] = (
                df_box_scores["equipo"]
                .str.replace(r"\s*\(.*\)?", "", regex=True)
                .str.strip()
            )

            player_to_team_map = pd.Series(
                df_box_scores["equipo_normalizado"].values,
                index=df_box_scores["NombreCompleto_limpio"]
            ).to_dict()

            # identificar equipos
            equipos = df_box_scores["equipo_normalizado"].unique()
            nombre_local = equipos[0]
            nombre_visitante = equipos[1] if len(equipos) > 1 else None

            return df_box_scores # Return the DataFrame if successful
        else:
            # If no player stats were found, df_box_scores remains None, so return None
            return None

    except Exception as e:
        print(f"Error procesando partido: {e}")
        return None # Return None on any exception
    finally:
        # 🔴 FUNDAMENTAL: cerrar navegador
        if driver is not None:
            driver.quit()


# --- PROCESAR TODOS LOS PARTIDOS ---
resultados_boxscore = [] # Lista original para Box Scores fusionados

llaves_partidos = list(match_data.keys())

for partido_key in tqdm(llaves_partidos, desc="Procesando partidos"):
    partido_link = match_data[partido_key]["link_estadisticas"]
    df_box_scores = extraer_boxscore(partido_link)

    if df_box_scores is None:
        continue

    resultado_final = df_box_scores.copy()
    # ... (Proceso de merge y limpieza de Box Score/Stats Avanzadas) ...

    resultado_final['partido_key'] = partido_key
    resultados_boxscore.append(resultado_final) # Se guarda el Box Score fusionado

# --- CONCATENAR AMBOS RESULTADOS ---
if resultados_boxscore:
    # Concatenación de Box Scores (como ya estaba)
    df_boxscore_final = pd.concat(resultados_boxscore, ignore_index=True)
    print("\n\u2705 DataFrame final con todos los registros de Box Scores.")
    # display(df_boxscore_final)

else:
    print("No se pudieron procesar partidos correctamente.")

Procesando partidos:   0%|          | 1/380 [00:07<47:35,  7.53s/it]

Error procesando partido: Extra data: line 1 column 181 (char 180)


Procesando partidos:   1%|          | 2/380 [00:14<44:54,  7.13s/it]

Error procesando partido: Extra data: line 1 column 183 (char 182)


Procesando partidos:   1%|          | 3/380 [00:20<41:48,  6.65s/it]

Error procesando partido: Extra data: line 1 column 183 (char 182)


... (744 outputs left out)

Procesando partidos: 100%|█████████▉| 379/380 [40:44<00:06,  6.16s/it]

Error procesando partido: Extra data: line 1 column 183 (char 182)


Procesando partidos: 100%|██████████| 380/380 [40:51<00:00,  6.45s/it]

Error procesando partido: Extra data: line 1 column 181 (char 180)
No se pudieron procesar partidos correctamente.


{'ATENAS (C) vs BOCA (007/10/2024 22:10)': {'nombre_local': 'ATENAS (C)',
  'puntos_local': '69',
  'nombre_visita': 'BOCA',
  'puntos_visita': '81',
  'link_estadisticas': 'https://estadisticascabb.gesdeportiva.es/partido/rM2-eTJQHJsR2FLJYE8GRw==?a=1'},
 'OBRAS vs PLATENSE (008/10/2024 20:00)': {'nombre_local': 'OBRAS',
  'puntos_local': '94',
  'nombre_visita': 'PLATENSE',
  'puntos_visita': '90',
  'link_estadisticas': 'https://estadisticascabb.gesdeportiva.es/partido/UYiBwuNnKVL9qCbXhOY76g==?a=1'},
 'INSTITUTO vs OLIMPICO (LB) (008/10/2024 21:00)': {'nombre_local': 'INSTITUTO',
  'puntos_local': '99',
  'nombre_visita': 'OLIMPICO (LB)',
  'puntos_visita': '73',
  'link_estadisticas': 'https://estadisticascabb.gesdeportiva.es/partido/Dwgh5U3-8hNWcMtXCbn6yw==?a=1'},
 'UNION (SF) vs SAN LORENZO (008/10/2024 21:30)': {'nombre_local': 'UNION (SF)',
  'puntos_local': '68',
  'nombre_visita': 'SAN LORENZO',
  'puntos_visita': '63',
  'link_estadisticas': 'https://estadisticascabb.gesdepor

funciones

In [ ]:
import unicodedata
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

# ---------- Utilidades (idénticas a las que venías usando) ----------
def quitar_tildes(texto):
    if isinstance(texto, str):
        return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

def normalizar_accion_simple(texto):
    s = str(texto).upper()
    s = quitar_tildes(s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def tiempo_a_segundos(t):
    try:
        hh, mm, ss = t.split(":")
        return int(hh) * 3600 + int(mm) * 60 + int(ss)
    except:
        return float('nan')

# ---------- Función de preprocesado por partido (basada en tu pipeline de 1 partido) ----------
def preprocesar_pbp_single(df_acciones_final, nombre_local, nombre_visitante):
    """Recibe df_acciones_final (raw del partido), nombre_local, nombre_visitante.
       Devuelve df_preprocesada listo para concatenar (misma estructura que tu chunk de 1 partido)."""
    df_preprocesada = df_acciones_final.copy()

    # Si no vienen los quintetos como listas en todas las filas, asumimos que tu función ya las genera correctamente.
    # A) crear jugadores_en_cancha si no existe (suma longitudes quintetos)
    def len_quinteto(x):
        try:
            return len(x) if x is not None else 0
        except Exception:
            return 0

    if 'jugadores_en_cancha' not in df_preprocesada.columns:
        df_preprocesada['jugadores_en_cancha'] = df_preprocesada.apply(
            lambda r: len_quinteto(r.get('quinteto_local')) + len_quinteto(r.get('quinteto_visita')), axis=1
        )

    # B) asignar equipo_accion y tipo_equipo_accion basado en jugador y quintetos (fallback a tipo existente)
    def asignar_equipo_accion(row):
        jugador = row.get('jugador')
        q_local = row.get('quinteto_local') or []
        q_vis = row.get('quinteto_visita') or []
        try:
            if pd.notna(jugador) and jugador in q_local:
                return nombre_local
            if pd.notna(jugador) and jugador in q_vis:
                return nombre_visitante
        except Exception:
            pass
        # fallback a tipo_equipo_accion si ya estaba
        if 'tipo_equipo_accion' in row and pd.notna(row.get('tipo_equipo_accion')):
            tipo = str(row.get('tipo_equipo_accion')).lower()
            if 'local' in tipo:
                return nombre_local
            if 'visit' in tipo:
                return nombre_visitante
        return row.get('equipo_accion', None)

    df_preprocesada['equipo_accion'] = df_preprocesada.apply(asignar_equipo_accion, axis=1)

    # tipo_equipo_accion: 'local' / 'visitante' según equipo_accion
    df_preprocesada['tipo_equipo_accion'] = df_preprocesada['equipo_accion'].apply(
        lambda x: 'local' if pd.notna(x) and str(x) == str(nombre_local) else ('visitante' if pd.notna(x) and str(x) == str(nombre_visitante) else None)
    )

    # 1) Filtrar por jugadores en cancha >= 10
    if 'jugadores_en_cancha' in df_preprocesada.columns:
        df_preprocesada = df_preprocesada[df_preprocesada["jugadores_en_cancha"] >= 10].copy()

    # 2) Normalizar acción
    df_preprocesada['accion_norm'] = df_preprocesada['accion'].apply(normalizar_accion_simple)

    # 3) Eliminar acciones prohibidas
    strings_a_eliminar = [
        "ENTRA A PISTA",
        "INICIO DEL PERIODO",
        "FINAL DEL PERIODO",
        "ABANDONA LA PISTA",
        "TAPON REALIZADO",
        "TECNICA",
        "TECNICA BANQUILLO (C)",
        "TECNICA BANQUILLO (B)",
        "TECNICA DESCALIFICANTE",
        "TECNICA DESCALIFICANTE ACOMPAÑANTE NO PARTICIPA",
        "TECNICA-E (ALINEACIÓN ILEGAL)",
        "TIEMPO MUERTO SOLICITADO",
        "ANTIDEPORTIVA"
    ]
    escaped_strings = [re.escape(s) for s in strings_a_eliminar]
    patron = "|".join(escaped_strings)
    df_preprocesada = df_preprocesada[~df_preprocesada['accion_norm'].str.contains(patron, regex=True, na=False)].copy()

    # 4) FLECHA DE ALTERNANCIA -> voltear equipos si aparece
    mask_flecha = df_preprocesada['accion_norm'].str.contains('FLECHA ALTERNANCIA', na=False)
    if mask_flecha.any():
        df_preprocesada.loc[mask_flecha, "equipo_accion"] = df_preprocesada.loc[mask_flecha, "equipo_accion"].map({
            nombre_local: nombre_visitante,
            nombre_visitante: nombre_local
        })
        df_preprocesada.loc[mask_flecha, "tipo_equipo_accion"] = df_preprocesada.loc[mask_flecha, "tipo_equipo_accion"].map({
            "local": "visitante",
            "visitante": "local"
        })

    # 5) Eliminar últimos REBOTE DEFENSIVO del inicio de cuarto si corresponde
    mask_last_in_quarter = df_preprocesada['cuarto'].fillna(-999) != df_preprocesada['cuarto'].shift(1).fillna(-999)
    mask_rebote_def = df_preprocesada['accion_norm'].str.contains(r'\bREBOTE DEFENSIVO\b', regex=True, na=False)
    mask_remove_last_rebote = mask_last_in_quarter & mask_rebote_def
    if mask_remove_last_rebote.any():
        df_preprocesada = df_preprocesada.loc[~mask_remove_last_rebote].copy()

    # 6) Reglas para faltas (usa accion previa en orden cronológico inverso como antes)
    accion_prev = df_preprocesada['accion_norm'].shift(-1).fillna('')
    equipo_prev = df_preprocesada['equipo_accion'].shift(-1)

    mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)?\b', regex=True, na=False)
    mask_perdida_prev = accion_prev.str.contains(r'\bPERDIDA\b', regex=True, na=False)
    mask_asistencia_prev = accion_prev.str.contains(r'\bASISTENCIA\b', regex=True, na=False)
    mask_triple_prev = accion_prev.str.contains(r'\bTRIPLE\b', na=False)
    mask_canasta2_prev = accion_prev.str.contains(r'\bCANASTA\b', na=False) & accion_prev.str.contains(r'\b2\b', na=False)

    tiempo_prev_seg = df_preprocesada['tiempo'].shift(-1).apply(tiempo_a_segundos)
    tiempo_curr_seg = df_preprocesada['tiempo'].apply(tiempo_a_segundos)
    elapsed = tiempo_prev_seg - tiempo_curr_seg

    mask_scoring_prev_diff_time = (
        (mask_triple_prev | mask_canasta2_prev | mask_asistencia_prev)
        & (elapsed > 4)
    )

    mask_prev_excepcion = (
        mask_tiro_prev
        | mask_perdida_prev
        | mask_scoring_prev_diff_time
    )

    mask_falta = df_preprocesada['accion_norm'] == 'FALTA COMETIDA'
    mask_equipo_distinto = df_preprocesada['equipo_accion'] != equipo_prev
    mask_keep_falta = mask_falta & mask_prev_excepcion & mask_equipo_distinto
    mask_a_eliminar = mask_falta & (~mask_keep_falta)
    if mask_a_eliminar.any():
        df_preprocesada = df_preprocesada.loc[~mask_a_eliminar].copy()

    # limpieza final: drop helper
    df_preprocesada = df_preprocesada.drop(columns=['accion_norm'])

    # 7) Añadir posesion, quintetos_por_pos, pts_acum, pts_fin
    # invertir a orden cronológico real (fila 0 = primera acción)
    df_tmp = df_preprocesada.iloc[::-1].reset_index(drop=True)

    cond_nueva_posesion = (
        (df_tmp['equipo_accion'] != df_tmp['equipo_accion'].shift(1)) |
        (df_tmp['cuarto'] != df_tmp['cuarto'].shift(1))
    )
    cond_nueva_posesion.iloc[0] = True
    df_tmp['posesion'] = cond_nueva_posesion.cumsum().astype(int)

    def contar_quintetos(gr):
        pares = list(zip(gr['quinteto_local'].fillna('__NA__'),
                         gr['quinteto_visita'].fillna('__NA__')))
        return len(pd.Series(pares).drop_duplicates())

    quintetos_por_pos = df_tmp.groupby('posesion').apply(contar_quintetos)
    quintetos_por_pos.name = 'quintetos_por_pos'
    df_tmp = df_tmp.join(quintetos_por_pos, on='posesion')

    # pts acumulados por forward fill (empiezan en 0)
    df_tmp['pts_acum_local'] = df_tmp['puntos_local'].ffill().fillna(0).astype(int)
    df_tmp['pts_acum_visit'] = df_tmp['puntos_visita'].ffill().fillna(0).astype(int)

    # pts_fin en última fila de cada posesión (en orden cronológico real)
    es_ultima_fila_posesion = df_tmp['posesion'] != df_tmp['posesion'].shift(-1)
    df_tmp['pts_fin_loc'] = None
    df_tmp['pts_fin_vis'] = None
    df_tmp.loc[es_ultima_fila_posesion, 'pts_fin_loc'] = df_tmp.loc[es_ultima_fila_posesion, 'pts_acum_local']
    df_tmp.loc[es_ultima_fila_posesion, 'pts_fin_vis'] = df_tmp.loc[es_ultima_fila_posesion, 'pts_acum_visit']

    # volver al orden original (igual que tu df original: última fila = última acción)
    df_preprocesada = df_tmp.iloc[::-1].reset_index(drop=True)

    return df_preprocesada

corrijo un error de carga

In [ ]:
# Eliminación manual de registro erróneo detectado en control de calidad
# Bug puntual del PBP:
# Partido: ZARATE BASKET vs ATENAS (C) (01/12/2024 21:00)
# Acción inválida: "ENTRA A PISTA EL#10" en 00:10:00 del 2° cuarto
#df_pbp_final = pd.read_csv("acciones_partido_crudo.csv")

# En R

df_pbp_final = pd.read_csv("data/df_pbp_final.csv")

Filas encontradas: 126
--- Vista previa de los datos cargados ---
                                           partido_key  cuarto    tiempo  \
61032  ZARATE BASKET vs ATENAS (C) (010/12/2024 21:00)     2.0  00:10:00   
61033  ZARATE BASKET vs ATENAS (C) (010/12/2024 21:00)     2.0  00:10:00   
61034  ZARATE BASKET vs ATENAS (C) (010/12/2024 21:00)     2.0  00:10:00   
61035  ZARATE BASKET vs ATENAS (C) (010/12/2024 21:00)     2.0  00:10:00   
61036  ZARATE BASKET vs ATENAS (C) (010/12/2024 21:00)     2.0  00:10:00   
61037  ZARATE BASKET vs ATENAS (C) (010/12/2024 21:00)     2.0  00:10:00   
61038  ZARATE BASKET vs ATENAS (C) (010/12/2024 21:00)     2.0  00:10:00   
61039  ZARATE BASKET vs ATENAS (C) (010/12/2024 21:00)     2.0  00:10:00   
61040  ZARATE BASKET vs ATENAS (C) (010/12/2024 21:00)     2.0  00:10:00   
61041  ZARATE BASKET vs ATENAS (C) (010/12/2024 21:00)     2.0  00:10:00   

                                                                                                 

toda la temporada - preprocesado

In [ ]:
import unicodedata
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

# ---------- Utilidades (idénticas a las que venías usando) ----------
def quitar_tildes(texto):
    if isinstance(texto, str):
        return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

def normalizar_accion_simple(texto):
    s = str(texto).upper()
    s = quitar_tildes(s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def tiempo_a_segundos(t):
    try:
        hh, mm, ss = t.split(":")
        return int(hh) * 3600 + int(mm) * 60 + int(ss)
    except:
        return float('nan')

# ---------- Función de preprocesado por partido (basada en tu pipeline de 1 partido) ----------
def preprocesar_pbp_single(df_acciones_final, nombre_local, nombre_visitante):
    """Recibe df_acciones_final (raw del partido), nombre_local, nombre_visitante.
       Devuelve df_preprocesada listo para concatenar (misma estructura que tu chunk de 1 partido)."""
    df_preprocesada = df_acciones_final.copy()

    # Si no vienen los quintetos como listas en todas las filas, asumimos que tu función ya las genera correctamente.
    # A) crear jugadores_en_cancha si no existe (suma longitudes quintetos)
    def len_quinteto(x):
        try:
            return len(x) if x is not None else 0
        except Exception:
            return 0

    if 'jugadores_en_cancha' not in df_preprocesada.columns:
        df_preprocesada['jugadores_en_cancha'] = df_preprocesada.apply(
            lambda r: len_quinteto(r.get('quinteto_local')) + len_quinteto(r.get('quinteto_visita')), axis=1
        )

    # B) asignar equipo_accion y tipo_equipo_accion basado en jugador y quintetos (fallback a tipo existente)
    def asignar_equipo_accion(row):
        jugador = row.get('jugador')
        q_local = row.get('quinteto_local') or []
        q_vis = row.get('quinteto_visita') or []
        try:
            if pd.notna(jugador) and jugador in q_local:
                return nombre_local
            if pd.notna(jugador) and jugador in q_vis:
                return nombre_visitante
        except Exception:
            pass
        # fallback a tipo_equipo_accion si ya estaba
        if 'tipo_equipo_accion' in row and pd.notna(row.get('tipo_equipo_accion')):
            tipo = str(row.get('tipo_equipo_accion')).lower()
            if 'local' in tipo:
                return nombre_local
            if 'visit' in tipo:
                return nombre_visitante
        return row.get('equipo_accion', None)

    df_preprocesada['equipo_accion'] = df_preprocesada.apply(asignar_equipo_accion, axis=1)

    # tipo_equipo_accion: 'local' / 'visitante' según equipo_accion
    df_preprocesada['tipo_equipo_accion'] = df_preprocesada['equipo_accion'].apply(
        lambda x: 'local' if pd.notna(x) and str(x) == str(nombre_local) else ('visitante' if pd.notna(x) and str(x) == str(nombre_visitante) else None)
    )

    # 1) Filtrar por jugadores en cancha >= 10
    if 'jugadores_en_cancha' in df_preprocesada.columns:
        df_preprocesada = df_preprocesada[df_preprocesada["jugadores_en_cancha"] >= 10].copy()

    # 2) Normalizar acción
    df_preprocesada['accion_norm'] = df_preprocesada['accion'].apply(normalizar_accion_simple)

    # 3) Eliminar acciones prohibidas
    strings_a_eliminar = [
        "ENTRA A PISTA",
        "INICIO DEL PERIODO",
        "FINAL DEL PERIODO",
        "ABANDONA LA PISTA",
        "TAPON REALIZADO",
        "TECNICA",
        "TECNICA BANQUILLO (C)",
        "TECNICA BANQUILLO (B)",
        "TECNICA DESCALIFICANTE",
        "TECNICA DESCALIFICANTE ACOMPAÑANTE NO PARTICIPA",
        "TECNICA-E (ALINEACIÓN ILEGAL)",
        "TIEMPO MUERTO SOLICITADO",
        "ANTIDEPORTIVA"
    ]
    escaped_strings = [re.escape(s) for s in strings_a_eliminar]
    patron = "|".join(escaped_strings)
    df_preprocesada = df_preprocesada[~df_preprocesada['accion_norm'].str.contains(patron, regex=True, na=False)].copy()

    # 4) FLECHA DE ALTERNANCIA -> voltear equipos si aparece
    mask_flecha = df_preprocesada['accion_norm'].str.contains('FLECHA ALTERNANCIA', na=False)
    if mask_flecha.any():
        df_preprocesada.loc[mask_flecha, "equipo_accion"] = df_preprocesada.loc[mask_flecha, "equipo_accion"].map({
            nombre_local: nombre_visitante,
            nombre_visitante: nombre_local
        })
        df_preprocesada.loc[mask_flecha, "tipo_equipo_accion"] = df_preprocesada.loc[mask_flecha, "tipo_equipo_accion"].map({
            "local": "visitante",
            "visitante": "local"
        })

    # 5) Eliminar últimos REBOTE DEFENSIVO del inicio de cuarto si corresponde
    mask_last_in_quarter = df_preprocesada['cuarto'].fillna(-999) != df_preprocesada['cuarto'].shift(1).fillna(-999)
    mask_rebote_def = df_preprocesada['accion_norm'].str.contains(r'\bREBOTE DEFENSIVO\b', regex=True, na=False)
    mask_remove_last_rebote = mask_last_in_quarter & mask_rebote_def
    if mask_remove_last_rebote.any():
        df_preprocesada = df_preprocesada.loc[~mask_remove_last_rebote].copy()

    # 6) Reglas para faltas (usa accion previa en orden cronológico inverso como antes)
    accion_prev = df_preprocesada['accion_norm'].shift(-1).fillna('')
    equipo_prev = df_preprocesada['equipo_accion'].shift(-1)

    mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)?\b', regex=True, na=False)
    mask_perdida_prev = accion_prev.str.contains(r'\bPERDIDA\b', regex=True, na=False)
    mask_asistencia_prev = accion_prev.str.contains(r'\bASISTENCIA\b', regex=True, na=False)
    mask_triple_prev = accion_prev.str.contains(r'\bTRIPLE\b', na=False)
    mask_canasta2_prev = accion_prev.str.contains(r'\bCANASTA\b', na=False) & accion_prev.str.contains(r'\b2\b', na=False)

    tiempo_prev_seg = df_preprocesada['tiempo'].shift(-1).apply(tiempo_a_segundos)
    tiempo_curr_seg = df_preprocesada['tiempo'].apply(tiempo_a_segundos)
    elapsed = tiempo_prev_seg - tiempo_curr_seg

    mask_scoring_prev_diff_time = (
        (mask_triple_prev | mask_canasta2_prev | mask_asistencia_prev)
        & (elapsed > 4)
    )

    mask_prev_excepcion = (
        mask_tiro_prev
        | mask_perdida_prev
        | mask_scoring_prev_diff_time
    )

    mask_falta = df_preprocesada['accion_norm'] == 'FALTA COMETIDA'
    mask_equipo_distinto = df_preprocesada['equipo_accion'] != equipo_prev
    mask_keep_falta = mask_falta & mask_prev_excepcion & mask_equipo_distinto
    mask_a_eliminar = mask_falta & (~mask_keep_falta)
    if mask_a_eliminar.any():
        df_preprocesada = df_preprocesada.loc[~mask_a_eliminar].copy()

    # limpieza final: drop helper
    df_preprocesada = df_preprocesada.drop(columns=['accion_norm'])

    # 7) Añadir posesion, quintetos_por_pos, pts_acum, pts_fin
    # invertir a orden cronológico real (fila 0 = primera acción)
    df_tmp = df_preprocesada.iloc[::-1].reset_index(drop=True)

    cond_nueva_posesion = (
        (df_tmp['equipo_accion'] != df_tmp['equipo_accion'].shift(1)) |
        (df_tmp['cuarto'] != df_tmp['cuarto'].shift(1))
    )
    cond_nueva_posesion.iloc[0] = True
    df_tmp['posesion'] = cond_nueva_posesion.cumsum().astype(int)

    def contar_quintetos(gr):
        pares = list(zip(gr['quinteto_local'].fillna('__NA__'),
                         gr['quinteto_visita'].fillna('__NA__')))
        return len(pd.Series(pares).drop_duplicates())

    quintetos_por_pos = df_tmp.groupby('posesion').apply(contar_quintetos)
    quintetos_por_pos.name = 'quintetos_por_pos'
    df_tmp = df_tmp.join(quintetos_por_pos, on='posesion')

    # pts acumulados por forward fill (empiezan en 0)
    df_tmp['pts_acum_local'] = df_tmp['puntos_local'].ffill().fillna(0).astype(int)
    df_tmp['pts_acum_visit'] = df_tmp['puntos_visita'].ffill().fillna(0).astype(int)

    # pts_fin en última fila de cada posesión (en orden cronológico real)
    es_ultima_fila_posesion = df_tmp['posesion'] != df_tmp['posesion'].shift(-1)
    df_tmp['pts_fin_loc'] = None
    df_tmp['pts_fin_vis'] = None
    df_tmp.loc[es_ultima_fila_posesion, 'pts_fin_loc'] = df_tmp.loc[es_ultima_fila_posesion, 'pts_acum_local']
    df_tmp.loc[es_ultima_fila_posesion, 'pts_fin_vis'] = df_tmp.loc[es_ultima_fila_posesion, 'pts_acum_visit']

    # volver al orden original (igual que tu df original: última fila = última acción)
    df_preprocesada = df_tmp.iloc[::-1].reset_index(drop=True)

    return df_preprocesada


resultados_pbp_preprocesados = []
llaves_partidos = list(match_data.keys())

for partido_key in tqdm(llaves_partidos, desc="Procesando partidos"):
    partido_link = match_data[partido_key].get("link_estadisticas")
    if not partido_link:
        print(f"Sin link para partido {partido_key}, salteando.")
        continue

    # extraer (usa tu función existente)
    df_box_scores, df_acciones_final = extraer_boxscore_y_pbp(partido_link)

    if df_acciones_final is None:
        print(f"No se extrajo PBP para {partido_key}, salteando.")
        continue

    # inferir nombres de equipos si es posible (desde boxscore)
    nombre_local = None
    nombre_visitante = None
    if df_box_scores is not None and 'equipo' in df_box_scores.columns:
        equipos_unicos = list(df_box_scores['equipo'].unique())
        if len(equipos_unicos) >= 1:
            nombre_local = equipos_unicos[0]
            nombre_visitante = equipos_unicos[1] if len(equipos_unicos) > 1 else None

    # aplicar preprocesado del partido
    try:
        df_pre_proc = preprocesar_pbp_single(df_acciones_final, nombre_local, nombre_visitante)
    except Exception as e:
        print(f"Error preprocesando partido {partido_key}: {e}")
        continue

    # añadir identificador del partido y guardar
    df_pre_proc['partido_key'] = partido_key
    resultados_pbp_preprocesados.append(df_pre_proc.copy())

# concatenar todos los pbp preprocesados
if resultados_pbp_preprocesados:
    df_pbp_final_preproc = pd.concat(resultados_pbp_preprocesados, ignore_index=True)
    print("✅ PBP preprocesado concatenado para los partidos procesados.")
    display(df_pbp_final_preproc.head())
else:
    df_pbp_final_preproc = pd.DataFrame()
    print("⚠️ No se preprocesó ninguna PBP.")


Procesando partidos:   0%|          | 0/380 [00:00<?, ?it/s]/tmp/ipython-input-3690761388.py:123: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)?\b', regex=True, na=False)
/tmp/ipython-input-3690761388.py:170: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  quintetos_por_pos = df_tmp.groupby('posesion').apply(contar_quintetos)
Procesando partidos:   0%|          | 1/380 [00:12<1:22:00, 12.98s/it]/tmp/ipython-input-3690761388.py:123: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extr

No se extrajo PBP para ZARATE BASKET vs INSTITUTO (023/01/2025 21:00), salteando.


/tmp/ipython-input-3690761388.py:123: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)?\b', regex=True, na=False)
/tmp/ipython-input-3690761388.py:170: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  quintetos_por_pos = df_tmp.groupby('posesion').apply(contar_quintetos)
Procesando partidos:  43%|████▎     | 165/380 [30:34<38:53, 10.86s/it]/tmp/ipython-input-3690761388.py:123: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)

No se extrajo PBP para ZARATE BASKET vs RIACHUELO (LR) (017/05/2025 20:30), salteando.


/tmp/ipython-input-3690761388.py:123: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)?\b', regex=True, na=False)
/tmp/ipython-input-3690761388.py:170: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  quintetos_por_pos = df_tmp.groupby('posesion').apply(contar_quintetos)
Procesando partidos:  98%|█████████▊| 373/380 [1:10:57<01:06,  9.48s/it]/tmp/ipython-input-3690761388.py:123: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(

✅ PBP preprocesado concatenado para los partidos procesados.


,cuarto,tiempo,accion,jugador,puntos_local,puntos_visita,quinteto_local,quinteto_visita,jugadores_en_cancha,jugador_limpio,equipo_accion,tipo_equipo_accion,posesion,quintetos_por_pos,pts_acum_local,pts_acum_visit,pts_fin_loc,pts_fin_vis,partido_key
0,4,00:00:21,PÉRDIDA DE BALÓN,"PROME, TIZIANO URIEL SEBASTIAN",NaN,NaN,"[ARAUJO, MAXIMO, LEMA, LEONARDO, MAIDANA, JERO...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"PROME, TIZIANO URIEL SEBASTIAN",BOCA,visitante,146,1,69,81,69,81,ATENAS (C) vs BOCA (007/10/2024 22:10)
1,4,00:00:32,REBOTE OFENSIVO#5,"GUERRERO MARGARIT, JUAN MARTIN",NaN,NaN,"[ARAUJO, MAXIMO, LEMA, LEONARDO, MAIDANA, JERO...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"GUERRERO MARGARIT, JUAN MARTIN",BOCA,visitante,146,1,69,81,None,None,ATENAS (C) vs BOCA (007/10/2024 22:10)
2,4,00:00:32,TIRO DE 3 FALLADO,"PROME, TIZIANO URIEL SEBASTIAN",NaN,NaN,"[ARAUJO, MAXIMO, LEMA, LEONARDO, MAIDANA, JERO...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"PROME, TIZIANO URIEL SEBASTIAN",BOCA,visitante,146,1,69,81,None,None,ATENAS (C) vs BOCA (007/10/2024 22:10)
3,4,00:00:55,TIRO LIBRE ANOTADO,"TOMATIS, TIAGO",69.0,81.0,"[ARAUJO, MAXIMO, LEMA, LEONARDO, MAIDANA, JERO...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"TOMATIS, TIAGO",ATENAS (C),local,145,1,69,81,69,81,ATENAS (C) vs BOCA (007/10/2024 22:10)
4,4,00:00:55,TIRO LIBRE ANOTADO,"TOMATIS, TIAGO",68.0,81.0,"[ARAUJO, MAXIMO, LEMA, LEONARDO, MAIDANA, JERO...","[CUELLO, MARTIN NICOLAS, DELIA, MARCOS NICOLAS...",10,"TOMATIS, TIAGO",ATENAS (C),local,145,1,68,81,None,None,ATENAS (C) vs BOCA (007/10/2024 22:10)


In [ ]:
df_pbp_final_preproc.to_csv("pbp_preprocesado_temporada.csv", index=False)

preprocesamiento a partir del crudo

In [ ]:
from tqdm import tqdm

resultados_pbp_preprocesados = []

# agrupar por partido (CLAVE)
for partido_key, df_partido in tqdm(
    df_pbp_final.groupby("partido_key"),
    desc="Preprocesando partidos"
):

    # seguridad: copiar
    df_partido = df_partido.copy()

    # inferir equipos desde el propio partido
    equipos = (
        df_partido["equipo_accion"]
        .dropna()
        .unique()
        .tolist()
    )

    nombre_local = equipos[0] if len(equipos) > 0 else None
    nombre_visitante = equipos[1] if len(equipos) > 1 else None

    try:
        df_pre = preprocesar_pbp_single(
            df_partido,
            nombre_local,
            nombre_visitante
        )
    except Exception as e:
        print(f"❌ Error en partido {partido_key}: {e}")
        continue

    df_pre["partido_key"] = partido_key
    resultados_pbp_preprocesados.append(df_pre)

# concatenar todo
if resultados_pbp_preprocesados:
    df_pbp_final_preproc = pd.concat(
        resultados_pbp_preprocesados,
        ignore_index=True
    )
    print("✅ PBP preprocesado completo.")
else:
    df_pbp_final_preproc = pd.DataFrame()
    print("⚠️ No se pudo preprocesar ningún partido.")


Preprocesando partidos:   0%|          | 0/378 [00:00<?, ?it/s]/tmp/ipython-input-4233366481.py:123: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_tiro_prev = accion_prev.str.contains(r'\bTIRO(S)?\b', regex=True, na=False)
/tmp/ipython-input-4233366481.py:170: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  quintetos_por_pos = df_tmp.groupby('posesion').apply(contar_quintetos)
Preprocesando partidos:   0%|          | 1/378 [00:00<00:54,  6.97it/s]/tmp/ipython-input-4233366481.py:123: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.

✅ PBP preprocesado completo.


In [ ]:
df_pbp_final_preproc.to_csv("pbp_preprocesado_temporada.csv", index=False)

toda la temporada - poss by poss

In [ ]:
# ---------------------------
# CREAR poss-by-poss concatenado para todos los partidos en df_pbp_final_preproc
# ---------------------------

import ast
import numpy as np
import pandas as pd

# ---- 0) chequeo rápido ----

if 'df_pbp_final_preproc' not in globals():
    raise RuntimeError("No encuentro 'df_pbp_final_preproc' en el entorno. Crea/concatena primero los PBP preprocesados.")

df_all = df_pbp_final_preproc.copy()

# ---- 1) parser robusto para quintetos (lista real, np.ndarray, string "['A','B']", None, etc.) ----
def parse_quinteto_python_list(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return [str(i).strip() for i in list(x) if pd.notna(i) and str(i).strip()!='']
    s = str(x).strip()
    if s.lower() in ('nan', 'none', ''):
        return []
    if s.startswith('[') and s.endswith(']'):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set, np.ndarray)):
                return [str(i).strip() for i in list(parsed) if pd.notna(i) and str(i).strip()!='']
        except Exception:
            pass
    return [s]

# ---- 2) construir lista UNIÓN de jugadores en todos los partidos (jugador + quintetos) ----
jugadores_from_col = df_all['jugador'].dropna().astype(str).unique().tolist()

todos_q = []
for q in pd.concat([df_all['quinteto_local'].fillna('[]'), df_all['quinteto_visita'].fillna('[]')], ignore_index=True):
    todos_q.extend(parse_quinteto_python_list(q))

jugadores_union = sorted(set(jugadores_from_col + todos_q))

# ---- 3) función para procesar un único partido (devuelve poss-by-poss de ese partido) ----
def procesar_partido_poss(df_partido):
    # asumimos que df_partido ya tiene columnas: 'posesion','pts_fin_loc','pts_fin_vis','quinteto_local','quinteto_visita',
    # 'equipo_accion','tipo_equipo_accion','equipo_local','equipo_visitante','jugador', etc.
    dfp = df_partido.copy().reset_index(drop=True)

    # quedarnos sólo con filas finales de posesión
    df_poss_rows = dfp[dfp['pts_fin_loc'].notna() | dfp['pts_fin_vis'].notna()].copy().reset_index(drop=True)
    if df_poss_rows.empty:
        return pd.DataFrame()  # no hay posesiones en este partido

    # parsear quintetos
    df_poss_rows['q_local_list'] = df_poss_rows['quinteto_local'].apply(parse_quinteto_python_list)
    df_poss_rows['q_vis_list']   = df_poss_rows['quinteto_visita'].apply(parse_quinteto_python_list)

    # asegurar numericidad
    for c in ['pts_fin_loc','pts_fin_vis','pts_acum_local','pts_acum_visit']:
        if c in df_poss_rows.columns:
            df_poss_rows[c] = pd.to_numeric(df_poss_rows[c], errors='coerce')

    # ordenar por posesion asc
    if 'posesion' in df_poss_rows.columns:
        df_poss_rows = df_poss_rows.sort_values('posesion').reset_index(drop=True)
    else:
        # si no hay posesion, crear una incremental (seguro que tu preprocesado la generó, pero por si acaso)
        df_poss_rows = df_poss_rows.reset_index(drop=True)
        df_poss_rows['posesion'] = np.arange(1, len(df_poss_rows)+1)

    # determinar si ataca local (preferir tipo_equipo_accion, sino comparar nombres de equipo)
    def ataca_local_row(r):
        tipo = r.get('tipo_equipo_accion')
        equipo_acc = r.get('equipo_accion')
        equipo_local = r.get('equipo_local')
        if isinstance(tipo, str) and tipo.strip() != '':
            t = tipo.lower()
            if 'local' in t:
                return True
            if 'visit' in t:
                return False
        return (pd.notna(equipo_acc) and pd.notna(equipo_local) and str(equipo_acc) == str(equipo_local))

    df_poss_rows['ataca_local'] = df_poss_rows.apply(ataca_local_row, axis=1)

    # preparar pts_fin rellenados y previous
    df_poss_rows['pts_fin_loc_f'] = df_poss_rows['pts_fin_loc'].fillna(0).astype(int)
    df_poss_rows['pts_fin_vis_f'] = df_poss_rows['pts_fin_vis'].fillna(0).astype(int)

    prev_pts_fin_loc = df_poss_rows['pts_fin_loc_f'].shift(1).fillna(0).astype(int)
    prev_pts_fin_vis = df_poss_rows['pts_fin_vis_f'].shift(1).fillna(0).astype(int)

    # calcular puntos por posesión según regla (diferencia con posesion anterior)
    df_poss_rows['puntos_pos'] = np.where(
        df_poss_rows['ataca_local'],
        (df_poss_rows['pts_fin_loc_f'] - prev_pts_fin_loc).astype(int),
        (df_poss_rows['pts_fin_vis_f'] - prev_pts_fin_vis).astype(int)
    ).astype(int)

    # determinar atacantes/defensores por fila (usar equipo_accion comparado con equipo_local/visitante preferentemente)
    def obtener_atac_def(row):
        q_local = set(row['q_local_list'])
        q_vis = set(row['q_vis_list'])
        equipo_acc = row.get('equipo_accion')
        equipo_local = row.get('equipo_local')
        equipo_visit = row.get('equipo_visitante')
        tipo_equipo_acc = row.get('tipo_equipo_accion', None)

        if pd.notna(equipo_acc) and pd.notna(equipo_local) and str(equipo_acc) == str(equipo_local):
            return q_local, q_vis
        if pd.notna(equipo_acc) and pd.notna(equipo_visit) and str(equipo_acc) == str(equipo_visit):
            return q_vis, q_local

        if isinstance(tipo_equipo_acc, str):
            t = tipo_equipo_acc.lower()
            if 'local' in t:
                return q_local, q_vis
            if 'visit' in t:
                return q_vis, q_local

        return q_local, q_vis

    # construir indicadores (usar jugadores_union para columnas estables)
    indic_rows = []
    for _, row in df_poss_rows.iterrows():
        atac, defen = obtener_atac_def(row)
        # inicializo en 0 para toda la unión de jugadores
        indic = {p: 0 for p in jugadores_union}
        for p in atac:
            if p in indic:
                indic[p] = 1
        for p in defen:
            if p in indic:
                indic[p] = -1
        indic_rows.append(indic)

    df_ind = pd.DataFrame(indic_rows, columns=jugadores_union)

    # concatenar: conservar todas las columnas originales + puntos_pos + indicadores
    cols_originales = [c for c in df_poss_rows.columns if c not in ('q_local_list','q_vis_list','pts_fin_loc_f','pts_fin_vis_f')]
    # asegurar columnas claves
    if 'puntos_pos' not in cols_originales:
        cols_originales = cols_originales + ['puntos_pos']
    if 'ataca_local' not in cols_originales:
        cols_originales = cols_originales + ['ataca_local']

    df_out = pd.concat([df_poss_rows[cols_originales].reset_index(drop=True), df_ind.reset_index(drop=True)], axis=1)

    return df_out

# ---- 4) procesar partido por partido y concatenar ----
dfs_poss = []
for partido_key, df_group in df_all.groupby('partido_key'):
    df_result = procesar_partido_poss(df_group)
    if df_result.empty:
        print(f"Advertencia: partido {partido_key} no generó posesiones.")
        continue
    # mantener partido_key (si se perdió)
    df_result['partido_key'] = partido_key
    dfs_poss.append(df_result)

if dfs_poss:
    df_poss_by_poss_all = pd.concat(dfs_poss, ignore_index=True, sort=False)
else:
    df_poss_by_poss_all = pd.DataFrame()

# ---- 5) limpieza final: ordenar columnas (info primero, luego jugadores) ----
if not df_poss_by_poss_all.empty:
    info_cols = [c for c in df_poss_by_poss_all.columns if c not in jugadores_union]
    df_poss_by_poss_all = df_poss_by_poss_all[info_cols + jugadores_union]

# ---- 6) mostrar / guardar ----
print("Poss-by-poss total filas:", len(df_poss_by_poss_all))
display(df_poss_by_poss_all.head(20))

# opcional: guardar
df_poss_by_poss_all.to_excel("poss_by_poss_temporada.xlsx", index=False)
# df_poss_by_poss_all.to_csv("poss_by_poss_10_partidos.csv", index=False)

Poss-by-poss total filas: 58625


,Unnamed: 0,cuarto,tiempo,accion,jugador,puntos_local,puntos_visita,quinteto_local,quinteto_visita,jugadores_en_cancha,jugador_limpio,equipo_accion,tipo_equipo_accion,partido_key,posesion,quintetos_por_pos,pts_acum_local,pts_acum_visit,pts_fin_loc,pts_fin_vis,puntos_acum_local_manual,puntos_acum_visitante_manual,ataca_local,puntos_pos,"AALIYA, LEE ABRAHAM","ACEVEDO, MAXIMILIANO JUNIORS","ACEVEDO, SEBASTIAN IGNACIO","ACTIS, JOAQUIN MATIAS","ACUÑA, ROBERTO SANTIAGO","AGUERRE, FEDERICO MATIAS","AGUIRRE, GUSTAVO NICOLAS","ALESSIO, JOSE IGNACIO JESUS","ALEXANDER, QUINTIN IMMANUEL","ALI, PRINCE ADAMS","ALIENDE, GUILLERMO ALFONZO","ALLENDE, TOMAS DANIEL","ALONSO, GONZALO NICOLAS","ALORDA, FRANCO","AMAN, GONZALO","AMPRIMO, ROMEO","ANDERSON, ALPHONSO JORDAN","ANDUJAR, LUCAS","APREA, JULIAN PABLO","ARAGON, JUAN LUCAS","ARAUJO SIRE, THOMAS","ARAUJO, MAXIMO","ARIAS, JUAN PABLO","ARN BUSTAMANTE, LUCAS MARTIN","ARN BUSTAMANTE, LUCAS MARTIN","ASCANIO SOLORZANO, JOSE GREGORIO","ASTULFI BOGLIC, SEGUNDO","AYALA, VALENTINO","AZPILICUETA PASQUALINI, ALEJO","BALLARENA, SANTIAGO","BANEGAS, JORGE EMMANUEL","BANYARD, NICHOLAS RYAN","BARCO, JEREMIAS","BARRAL, PEDRO","BARRALES, SANTIAGO","BARREIRO, AGUSTIN","BARRIONUEVO, FELIPE","BARRIOS, LAUTARO","BARROS, AGUSTIN","BASABE, EMILIANO GUILLERMO","BASTARDO JOSE, RAYMON SCARLIN","BASUALDO, IVAN NICOLAS","BECTON, REGINALD GEQUAN","BEDNAREK, IGNACIO NICOLAS","BELL, RANDY TYREE","BENAY, LORENZO MARIANO","BENDER, VIKTOR DANIEL","BERNABEI, LISANDRO","BETTIGA, LEONEL VALENTIN","BOMBINO PARADA, PEDRO ANTONIO","BOND JR, TIMOTHY LAMONT","BONIZIOLI HONORATO, ITALO","BORSATTI, MATIAS DANIEL","BOTTA, TOMAS","BOWIE JR, JULIUS R","BROCAL, AGUSTIN ANDRES","BRUSSINO, JUAN IGNACIO","BUEMO, CARLOS EMANUEL","BUENDIA, CARLOS MANUEL","BUYATTI, MATEO","CABOT DOMINGUEZ, MARCOS","CABRERA, MARTIN ANDRES","CAFFARO, ESTEBAN","CAGLIERO, MATIAS","CALFANI PERSINCULA, MATHIAS KENY","CAPELLI, SANTIAGO","CAPORALETTI, MARTINO","CAPPONI, FEBO LORENZO","CARABALI POROZO, BRYAN JOSE","CARDENAS ZAMORA, JORDAN ISRAEL","CARDO, CRISTIAN","CARRERAS PEGUERO, XAVIER MANUEL","CARTER, MYLES JUSTIN","CECI, MATEO","CERINO, IAN","CHACON TIRADO, MARCOS","CHAMORRO, FACUNDO","CHAPERO, MARTIN","CHAPERO, TOMAS","CHIARAVIGLIO, NICOLAS","CIFUENTES ROTTA, FIDEL","CISNEROS SIERRA, ANYELO JOSE","CLARKE, CHRISTOPHER ASHTON","COGLIATI, PATRICIO ANDRES","COLLOMB, DIEGO","CONRRADI, FRANCISCO","COOPER, THOMAS","COPELLO, NICOLAS","COPESKY, JOSE IGNACIO","CORBALAN, JUAN PABLO","CORDOBA, GASTON EZEQUIEL","CORSI, BAUTISTA","CORSO SOTERAS, MATEO LEANDRO","COSOLITO, MAURO NICOLAS","COSTA, ALBANO NAHUEL","COSTA, ALBANO NAHUEL","CRAION JR, MICHAEL JOE","CUELLO, MARTIN NICOLAS","DANIELS, DEANDRE MARTISE","DANIELS, TRAVIS DORREL","DATO, MARTINIANO","DE LOS SANTOS, NICOLAS","DEFELIPPO, JOSE","DEL DO, FACUNDO","DELIA, MARCOS NICOLAS","DELIA, TOMAS FRANCISCO","DI PRINZIO, PIERO","DIAZ GIMENEZ, JORGE LUCIANO","DIAZ, FABRICIO HERNAN","DIEZ, ALEJANDRO GABRIEL","DIGGS, AVERY GERELL","DIGON, ALEJO ADRIEL","DOMINICI, LUCIO","DONZINO, OCTAVIO","DROCEZESKY, TIAGO IGNACIO","DUPUY, EZEQUIEL MARIO","ELIAS, GASTON FEDERICO","ESPINOZA, PABLO ANIBAL","FACELLO, AGUSTIN PEDRO","FARIAS ALLAMANO, JUAN ANGEL VALENTIN","FERGUSON, ROMEAO VENILL","FERNANDEZ ALVAREZ, JOAQUIN","FERNANDEZ, VICTOR LUIS","FERRARO, FRANCISCO","FERRI, VALENTINO","FIELDS, CALEB JOSEPH","FIERRO, MARIANO","FIGUEREDO, DIEGO RAFAEL","FILIPPETTI SATURNO, ENZO MATEO","FINETTI LOPEZ, VALENTINO","FLOR, ERIC FABIAN","FORESTIER, VALENTIN","FRAGA PEREZ, LAUTARO EZEQUIEL","FRANCA DOS SANTOS, JOAO VITOR","FRANCHELA, TOBIAS","FRANCO, NICOLAS ARIEL","FRONTERA, RAMIRO JEREMIAS","FUNDORA ARRECHAVALETA, YASMANY","GALLEGOS, RODRIGO AGUSTIN","GARCIA GUERRERO, JORGE LUIS","GARCIA, GASTON LEONEL","GARELLO, VICENTE","GARGALLO, LUCAS HERNAN","GENERO, JERONIMO RUBEN","GENITTI, MAXIMO","GENNERO, AUGUSTO","GILETTO, SALVADOR","GIORGETTI, FRANCO NAHUEL","GIULIANI, LUCCA LORENZO","GIVENS, SAMUEL JAMAL","GOICOECHEA, MARIANO EZEQUIEL","GOMEZ QUINTEROS, CAMILO 

In [ ]:
# opcional: guardar
#df_poss_by_poss_all.to_excel("poss_by_poss_temporada.xlsx", index=False)
df_poss_by_poss_all.to_csv("poss_by_poss_temporada.csv", index=False)